# 📄 مشروع وكيل تحليل وتطوير السيرة الذاتية الذكي (AI CV Agent)
### تحليل السير الذاتية بالذكاء الاصطناعي باستخدام LangChain و OpenAI و Streamlit

**فكرة المشروع:** يرفع المستخدم سيرته الذاتية، فيحللها النظام عبر خمسة وكلاء ذكيين، ويعرض له الفجوة المهارية
والوظائف والدورات المناسبة، ويعيد صياغة سيرته باحترافية — بواجهة تدعم **العربية والإنجليزية**.

| الميزة | الوصف |
|---|---|
| قراءة الملفات | PDF و Word (بما فيها الجداول) مع تنظيف النص، وإصلاح النص العربي المعكوس والحروف المتصلة، والتنبيه على الملفات الممسوحة ضوئياً |
| الحد من الهلوسة | قواعد صارمة في تعليمات كل وكيل، وتحقق بعد النموذج يحذف المهارات والاسم والأرقام غير الموجودة في السيرة |
| تحليل السيرة | استخراج الاسم والمسمى والمجال والمؤهل والمهارات وسنوات الخبرة |
| الفجوة المهارية | المهارات الناقصة مرتبة حسب الأولوية مع سبب كل مهارة |
| الوظائف | مسميات مقترحة حسب المدينة مع نسبة مطابقة وروابط بحث جاهزة |
| الدورات | دورة لكل مهارة ناقصة مع روابط Coursera و Udemy و LinkedIn Learning |
| تحسين السيرة | ملخص احترافي ونقاط خبرة معاد صياغتها ونصائح ATS |
| اللغتان | زر تبديل عربي / English للواجهة والنتائج والتقرير |
| عدّاد الزوار | عدد الزوار الكلي وزوار اليوم بالإنجليزية في التذييل |
| فحص النظام | صفحة داخل الواجهة تشغّل 10 اختبارات وتعرض نتيجتها بضغطة زر |

**معمارية النظام:**

```
المستخدم (Streamlit) ──► قراءة الملف (PDF / Word) ──► تنظيف النص
                                                          │
                     ┌────────────────────────────────────┴───────────┐
                     ▼                                                ▼
           وكيل استخراج البيانات                               وكيل تحسين السيرة
                     │
            ┌────────┴─────────┐
            ▼                  ▼
   وكيل الفجوة المهارية      وكيل الوظائف ──► روابط LinkedIn / Indeed / Bayt / Tanqeeb / جدارات
            │
            ▼
      وكيل الدورات ──► روابط Coursera / Udemy / LinkedIn Learning
            │
            ▼
   النتائج المنظمة (Pydantic) ──► الواجهة باللغتين + تقرير قابل للتحميل

صفحة «فحص النظام» ──► checks.py ──► تختبر كل الملفات أعلاه وتعرض النتيجة داخل الواجهة
```

**هيكل الملفات:**

| الملف | الدور |
|---|---|
| `i18n.py` | نصوص الواجهة والتقرير باللغتين، وأسماء المدن |
| `schemas.py` | هياكل البيانات (Pydantic) لمخرجات الوكلاء |
| `file_reader.py` | قراءة PDF و Word وتنظيف النص |
| `agents.py` | النموذج اللغوي والوكلاء الخمسة (Prompt Templates + LCEL) |
| `report.py` | روابط البحث عن الوظائف والدورات، وتقرير Markdown |
| `pipeline.py` | سير العمل: تشغيل الوكلاء على ثلاث مراحل متوازية |
| `sample_data.py` | سيرة ونتائج تجريبية يستخدمها «فحص النظام» |
| `checks.py` | اختبارات صفحة «فحص النظام» داخل الواجهة |
| `assets.py` | شعار الكلية التطبيقية بجامعة الأمير مساعد بن عبدالرحمن |
| `visitors.py` | عدّاد الزوار (الكلي واليوم) |
| `app.py` | واجهة Streamlit بصفحتين: التحليل وفحص النظام (تصميم Figma من قالب Simple Design System) |

**طريقة التشغيل:** شغّل الخلايا بالترتيب من الأعلى للأسفل. كل جزئية تكتب ملفها الخاص (`%%writefile`)،
وفي النهاية يُشغَّل تطبيق Streamlit. **الاختبارات موجودة داخل الواجهة** في صفحة «فحص النظام».



> **المفاتيح:** أضف في Colab Secrets (أيقونة المفتاح 🔑 في القائمة الجانبية) المفتاح `OPENAI_API_KEY`
> وفعّل "Notebook access". لا توجد أي مفاتيح داخل الكود.
> اختياري: `OPENAI_MODEL` لتغيير النموذج (الافتراضي `gpt-4o-mini`). خانة النموذج مخفية في الواجهة، فالتغيير يكون من هنا فقط.

## 0) تثبيت المكتبات (مرة واحدة فقط)

In [1]:
!pip install -qU langchain-core langchain-openai pydantic pymupdf python-docx streamlit

## 1) تحميل المفاتيح من Colab Secrets

In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
try:
    os.environ["OPENAI_MODEL"] = userdata.get("OPENAI_MODEL")
except Exception:
    pass  # سيُستخدم النموذج الافتراضي gpt-4o-mini
print("✅ تم تحميل المفاتيح بأمان")

✅ تم تحميل المفاتيح بأمان


## 2) نصوص اللغتين — `i18n.py`
كل نص يظهر في الواجهة أو التقرير موجود هنا بالعربية والإنجليزية، مع قائمة المدن.

In [ ]:
%%writefile i18n.py
"""
i18n.py — جميع نصوص الواجهة والتقرير باللغتين (العربية / English)
"""
from typing import Dict, List, Optional, Tuple

LANGS = ("ar", "en")

# المدن: المفتاح ← (الاسم العربي، الاسم الإنجليزي لروابط البحث). None = كل المملكة
CITIES: Dict[str, Tuple[str, str, Optional[str]]] = {
    "riyadh": ("الرياض", "Riyadh", "Riyadh"),
    "jeddah": ("جدة", "Jeddah", "Jeddah"),
    "dammam": ("الدمام", "Dammam", "Dammam"),
    "khobar": ("الخبر", "Khobar", "Khobar"),
    "makkah": ("مكة المكرمة", "Makkah", "Makkah"),
    "madinah": ("المدينة المنورة", "Madinah", "Madinah"),
    "khamis": ("خميس مشيط", "Khamis Mushait", "Khamis Mushait"),
    "abha": ("أبها", "Abha", "Abha"),
    "tabuk": ("تبوك", "Tabuk", "Tabuk"),
    "neom": ("نيوم", "NEOM", "NEOM"),
    "all": ("كل المملكة", "All of Saudi Arabia", None),
}


def city_name(key: str, lang: str) -> str:
    ar, en, _ = CITIES[key]
    return ar if lang == "ar" else en


def city_search(key: str) -> Optional[str]:
    return CITIES[key][2]


TEXT: Dict[str, Dict[str, object]] = {
    "ar": {
        # الصفحات
        "nav": ["التحليل", "فحص النظام"],
        "subtitle_hero": "خمسة وكلاء ذكيين يحللون مهاراتك، ويكشفون الفجوة، ويقترحون الوظائف والدورات، ويعيدون صياغة سيرتك.",
        "cta_upload": "ارفع سيرتك",
        "how_title": "كيف يعمل؟",
        "steps": [["ارفع سيرتك", "PDF أو Word"], ["يحللها الوكلاء الخمسة", "مهارات، فجوة، وظائف، دورات"],
                  ["استلم خطة تطويرك", "مع تقرير قابل للتحميل"]],
        "panel_title": "ابدأ التحليل",
        # فحص النظام
        "chk_title": "فحص النظام",
        "chk_sub": "تأكد إن كل أجزاء المشروع شغالة قبل العرض — بضغطة زر.",
        "chk_run": "تشغيل الفحص",
        "chk_total": "إجمالي الاختبارات",
        "chk_passed": "نجحت",
        "chk_failed": "فشلت",
        "chk_time": "المدة",
        "chk_sec": "ث",
        "chk_status": {"pass": "نجح", "fail": "فشل", "skip": "تخطّي"},
        "chk_empty": "اضغط «تشغيل الفحص» لاختبار كل أجزاء النظام.",
        "chk_note": "اختبار الاتصال بـ OpenAI يحتاج مفتاحاً صالحاً، وباقي الاختبارات تعمل بدون إنترنت.",
        "chk_names": {"i18n": "نصوص اللغتين", "schemas": "هياكل البيانات", "word": "قراءة ملفات Word",
                      "pdf": "قراءة ملفات PDF", "unsupported": "رفض الصيغ غير المدعومة",
                      "links": "روابط الوظائف والدورات", "agents": "بناء الوكلاء",
                      "pipeline": "سير العمل (تجريبي)", "arabic": "تصحيح النص العربي",
                      "openai": "الاتصال بـ OpenAI"},
        "chk_d_i18n": "كل النصوص موجودة بالعربية والإنجليزية ({n} نصاً)",
        "chk_d_schemas": "التصحيح التلقائي يعمل: 150% تصبح 100% وخبرة سالبة تصبح 0",
        "chk_d_word": "قراءة الفقرات والجداول من ملف تجريبي ({n} حرفاً)",
        "chk_d_pdf": "إنشاء PDF تجريبي واستخراج نصه ({n} حرفاً)",
        "chk_d_unsupported": "رسالة خطأ واضحة عند رفع ملف .txt",
        "chk_d_links": "{n} روابط بحث صحيحة",
        "chk_d_agents": "الوكلاء الخمسة جاهزون ({n})",
        "chk_d_pipeline": "المراحل الثلاث تعمل بالتوازي ببيانات تجريبية",
        "chk_d_openai": "النموذج {model} استجاب: {reply}",
        "chk_d_arabic": "النص العربي المعكوس والحروف المتصلة تُصحَّح، والمهارات غير الموجودة في السيرة تُحذف",
        "chk_d_openai_skip": "لا يوجد مفتاح OpenAI — تم التخطي",
        "api_key_label": "مفتاح OpenAI (لا يُحفظ)",
        "no_key": "الخدمة غير مهيأة بعد: يجب إضافة مفتاح OPENAI_API_KEY في إعدادات التطبيق (Secrets).",
        # الشريط الجانبي
        "page_title": "وكيل السيرة الذاتية",
        "brand_sub": "وكيل السيرة الذاتية الذكي",
        "language": "اللغة",
        "settings": "الإعدادات",
        "api_ok": "مفتاح OpenAI متصل",
        "api_input": "مفتاح OpenAI API",
        "api_missing": "أدخل المفتاح لتفعيل التحليل.",
        "model": "النموذج",
        "city": "المدينة",
        "footer": "المفتاح لا يُحفظ ويُستخدم لهذه الجلسة فقط.",
        # الرأس والرفع
        "tag": "مدعوم بالذكاء الاصطناعي",
        "uni_name": "جامعة الأمير مساعد بن عبدالرحمن",
        "college": "الكلية التطبيقية · دبلوم الذكاء الاصطناعي التوليدي",
        "hero_stats": ["5 وكلاء ذكيين", "3 مراحل متوازية", "عربي / English"],
        "footer_project": "وكيل تطوير السيرة الذاتية الذكي — المشروع النهائي لمقرر أساسيات LangChain",
        "footer_sup": "إشراف د. محمد الشهري",
        "footer_year": "العام الدراسي 1448هـ",
        "title": "حلّل سيرتك الذاتية واكتشف فرصك",
        "subtitle": "تحليل المهارات والفجوة المهارية، ووظائف ودورات مقترحة، وإعادة صياغة احترافية لسيرتك.",
        "upload": "ارفع سيرتك الذاتية — PDF أو Word (.docx)",
        "analyze": "ابدأ التحليل",
        "reanalyze": "أعد التحليل",
        "analyzing": "جاري التحليل…",
        "done": "اكتمل التحليل",
        "stale_file": "النتائج المعروضة لملف سابق. اضغط زر التحليل لتحليل الملف الحالي.",
        "stale_settings": "غيّرت اللغة أو المدينة أو النموذج. اضغط زر التحليل لتحديث النتائج.",
        "truncated": "السيرة طويلة جداً فتم تحليل الجزء الأول منها فقط.",
        # الأرقام
        "m_name": "الاسم",
        "m_years": "سنوات الخبرة",
        "m_skills": "المهارات المستخرجة",
        "m_top": "أعلى مطابقة",
        "tabs": ["الملف", "الفجوة المهارية", "الوظائف", "الدورات", "تحسين السيرة"],
        # الملف
        "current_title": "المسمى الحالي",
        "field": "المجال",
        "education": "المؤهل",
        "skills": "المهارات المستخرجة",
        "extracted_text": "عرض النص المستخرج من الملف",
        # الفجوة
        "gap_heading": "الفجوة المهارية",
        "tips_heading": "نصائح للتطوير",
        "priority": {"high": "عالية", "medium": "متوسطة", "low": "منخفضة"},
        # الوظائف
        "jobs_heading": "الوظائف المقترحة في {city}",
        "jobs_note": "الجهات المذكورة أمثلة للاسترشاد وليست شواغر مؤكدة.",
        "match": "مطابقة",
        "employers": "جهات للاسترشاد:",
        "jadarat": "جدارات",
        # الدورات
        "courses_heading": "الدورات المقترحة",
        "covers": "تغطي مهارة:",
        "level": {"beginner": "مبتدئ", "intermediate": "متوسط", "advanced": "متقدم"},
        # تحسين السيرة
        "summary_label": "الملخص الاحترافي الجديد (قابل للنسخ)",
        "bullets_heading": "نقاط الخبرة بعد إعادة الصياغة",
        "bullets_note": "الرمز [X] يعني رقماً تضيفه أنت من واقع خبرتك. النص مكتوب بلغة سيرتك الأصلية.",
        "ats_heading": "نصائح لأنظمة الفرز الآلي (ATS)",
        "download": "تحميل التقرير",
        # مراحل التحليل
        "step_1": "استخراج بيانات المرشح وإعادة صياغة السيرة…",
        "step_2": "تحليل الفجوة المهارية والبحث عن الوظائف المناسبة…",
        "step_3": "ترشيح الدورات التدريبية…",
        # الأخطاء
        "err_unsupported": "صيغة الملف غير مدعومة. ارفع ملف PDF أو Word (.docx).",
        "err_corrupt": "تعذرت قراءة الملف، قد يكون تالفاً أو محمياً بكلمة مرور.",
        "err_scanned": "النص المستخرج قليل جداً. غالباً الملف صورة ممسوحة ضوئياً؛ صدّر السيرة كـ PDF نصي أو ارفع ملف .docx.",
        "err_auth": "مفتاح OpenAI غير صحيح أو منتهي.",
        "err_rate": "تجاوزت حد الاستخدام أو الرصيد في OpenAI. حاول لاحقاً.",
        "err_model": "النموذج '{model}' غير متاح لحسابك. غيّر OPENAI_MODEL في Colab Secrets.",
        "err_unexpected": "حدث خطأ غير متوقع:",
        # التقرير
        "r_title": "تقرير تحليل السيرة الذاتية",
        "r_info": "المعلومات الأساسية",
        "r_search": "ابحث:",
        "r_truncated": "ملاحظة: السيرة طويلة جداً فتم تحليل أول {n} حرف فقط.",
    },
    "en": {
        "page_title": "CV Agent",
        "nav": ["Analysis", "System check"],
        "subtitle_hero": "Five AI agents analyze your skills, reveal the gap, suggest jobs and courses, and rewrite your CV.",
        "cta_upload": "Upload your CV",
        "how_title": "How it works",
        "steps": [["Upload your CV", "PDF or Word"], ["Five agents analyze it", "Skills, gap, jobs, courses"],
                  ["Get your growth plan", "With a downloadable report"]],
        "panel_title": "Start the analysis",
        "chk_title": "System check",
        "chk_sub": "Make sure every part of the project works before presenting — in one click.",
        "chk_run": "Run checks",
        "chk_total": "Total checks",
        "chk_passed": "Passed",
        "chk_failed": "Failed",
        "chk_time": "Duration",
        "chk_sec": "s",
        "chk_status": {"pass": "Passed", "fail": "Failed", "skip": "Skipped"},
        "chk_empty": "Click “Run checks” to test every part of the system.",
        "chk_note": "The OpenAI connection check needs a valid key; all other checks run offline.",
        "chk_names": {"i18n": "Bilingual texts", "schemas": "Data schemas", "word": "Reading Word files",
                      "pdf": "Reading PDF files", "unsupported": "Rejecting unsupported files",
                      "links": "Job and course links", "agents": "Building the agents",
                      "pipeline": "Pipeline (sample data)", "arabic": "Arabic text repair",
                      "openai": "OpenAI connection"},
        "chk_d_i18n": "Every text exists in Arabic and English ({n} texts)",
        "chk_d_schemas": "Auto-correction works: 150% becomes 100% and negative experience becomes 0",
        "chk_d_word": "Read paragraphs and tables from a sample file ({n} characters)",
        "chk_d_pdf": "Created a sample PDF and extracted its text ({n} characters)",
        "chk_d_unsupported": "Clear error message for a .txt upload",
        "chk_d_links": "{n} valid search links",
        "chk_d_agents": "All five agents are ready ({n})",
        "chk_d_pipeline": "All three stages run in parallel with sample data",
        "chk_d_openai": "Model {model} replied: {reply}",
        "chk_d_arabic": "Reversed Arabic and presentation-form letters are repaired; skills not in the CV are removed",
        "chk_d_openai_skip": "No OpenAI key — skipped",
        "api_key_label": "OpenAI key (never stored)",
        "no_key": "The service isn't configured yet: add OPENAI_API_KEY in the app settings (Secrets).",
        "brand_sub": "Smart CV Agent",
        "language": "Language",
        "settings": "Settings",
        "api_ok": "OpenAI key connected",
        "api_input": "OpenAI API key",
        "api_missing": "Enter your key to enable the analysis.",
        "model": "Model",
        "city": "City",
        "footer": "Your key is never stored; it's used for this session only.",
        "tag": "AI-powered",
        "uni_name": "Prince Musaid bin Abdulrahman University",
        "college": "Applied College · Generative AI Diploma",
        "hero_stats": ["5 AI agents", "3 parallel stages", "Arabic / English"],
        "footer_project": "Smart CV Development Agent — LangChain Fundamentals final project",
        "footer_sup": "Supervised by Dr. Mohammed Al-Shahri",
        "footer_year": "Academic year 1448 AH",
        "title": "Analyze your CV and discover your opportunities",
        "subtitle": "Skills and gap analysis, suggested jobs and courses, and a professional rewrite of your CV.",
        "upload": "Upload your CV — PDF or Word (.docx)",
        "analyze": "Start analysis",
        "reanalyze": "Analyze again",
        "analyzing": "Analyzing…",
        "done": "Analysis complete",
        "stale_file": "These results belong to a previous file. Click analyze to process the current one.",
        "stale_settings": "You changed the language, city or model. Click analyze to refresh the results.",
        "truncated": "Your CV is very long, so only its first part was analyzed.",
        "m_name": "Name",
        "m_years": "Years of experience",
        "m_skills": "Skills extracted",
        "m_top": "Top match",
        "tabs": ["Profile", "Skill gap", "Jobs", "Courses", "Improved CV"],
        "current_title": "Current title",
        "field": "Field",
        "education": "Education",
        "skills": "Extracted skills",
        "extracted_text": "Show the text extracted from the file",
        "gap_heading": "Skill gap",
        "tips_heading": "How to close the gap",
        "priority": {"high": "High", "medium": "Medium", "low": "Low"},
        "jobs_heading": "Suggested jobs in {city}",
        "jobs_note": "Listed employers are examples for guidance, not confirmed openings.",
        "match": "match",
        "employers": "Example employers:",
        "jadarat": "Jadarat",
        "courses_heading": "Suggested courses",
        "covers": "Covers:",
        "level": {"beginner": "Beginner", "intermediate": "Intermediate", "advanced": "Advanced"},
        "summary_label": "New professional summary (ready to copy)",
        "bullets_heading": "Rewritten experience bullets",
        "bullets_note": "[X] marks a number you fill in from your own experience. Text is written in your CV's original language.",
        "ats_heading": "Tips for applicant tracking systems (ATS)",
        "download": "Download report",
        "step_1": "Extracting candidate data and rewriting the CV…",
        "step_2": "Analyzing the skill gap and matching jobs…",
        "step_3": "Recommending courses…",
        "err_unsupported": "Unsupported file type. Upload a PDF or Word (.docx) file.",
        "err_corrupt": "Couldn't read the file; it may be corrupted or password-protected.",
        "err_scanned": "Very little text was extracted. The file is probably a scanned image; export your CV as a text PDF or upload a .docx.",
        "err_auth": "Your OpenAI key is invalid or expired.",
        "err_rate": "You've hit your OpenAI rate limit or run out of credit. Try again later.",
        "err_model": "The model '{model}' isn't available to your account. Change OPENAI_MODEL in Colab Secrets.",
        "err_unexpected": "Unexpected error:",
        "r_title": "CV Analysis Report",
        "r_info": "Basic information",
        "r_search": "Search:",
        "r_truncated": "Note: your CV is very long, so only the first {n} characters were analyzed.",
    },
}


def t(lang: str, key: str, **kwargs) -> object:
    value = TEXT.get(lang, TEXT["ar"])[key]
    return value.format(**kwargs) if kwargs and isinstance(value, str) else value


def tabs(lang: str) -> List[str]:
    return list(TEXT[lang]["tabs"])


## 3) هياكل البيانات — `schemas.py`
تحدد شكل مخرجات كل وكيل (Output Parsers عبر Pydantic)، مع تصحيح تلقائي للأرقام غير المنطقية.

In [ ]:
%%writefile schemas.py
"""
schemas.py — هياكل البيانات (Pydantic) التي يلتزم بها النموذج في مخرجاته.
الأوصاف محايدة لغوياً؛ لغة المخرجات تُحدد في تعليمات الوكلاء.
"""
from typing import List, Literal

from pydantic import BaseModel, Field, field_validator


class CandidateInfo(BaseModel):
    name: str = Field(description="Candidate's full name copied exactly as written in the CV (same words, same order), or 'N/A'")
    current_title: str = Field(description="Current or closest job title, in the output language")
    field: str = Field(description="General professional field, e.g. data analysis, accounting, nursing")
    education: str = Field(description="Highest degree and major, or 'N/A'")
    summary: str = Field(description="2-3 sentence summary of the CV")
    extracted_skills: List[str] = Field(description="Technical and soft skills explicitly mentioned in the CV only, "
                                                     "copied exactly as written in the CV (same language and spelling)")
    experience_years: float = Field(description="Total years of actual work experience; 0 for fresh graduates")

    @field_validator("experience_years")
    @classmethod
    def _clamp_years(cls, v: float) -> float:
        return round(min(max(v, 0.0), 60.0), 1)


class MissingSkill(BaseModel):
    skill: str = Field(description="Missing skill name (technical terms in English)")
    priority: Literal["high", "medium", "low"] = Field(description="Learning priority")
    reason: str = Field(description="One sentence on why this skill matters for this candidate in the Saudi market")


class SkillGap(BaseModel):
    missing_skills: List[MissingSkill] = Field(description="3 to 6 missing skills ordered by priority")
    recommendations_to_improve: List[str] = Field(description="Short practical tips to close the gap")


class JobMatch(BaseModel):
    job_title: str = Field(description="Job title in English exactly as used in job ads (short, searchable)")
    job_title_local: str = Field(description="The same job title in the output language")
    match_percentage: int = Field(description="Realistic match percentage between 0 and 100")
    why_fit: str = Field(description="One sentence on why this job fits the candidate")
    example_employers: List[str] = Field(
        description="2-3 types or examples of employers that usually hire for this title in the city "
                    "(guidance only, not confirmed openings)"
    )

    @field_validator("match_percentage")
    @classmethod
    def _clamp_pct(cls, v: int) -> int:
        return int(min(max(v, 0), 100))


class JobRecommendations(BaseModel):
    jobs: List[JobMatch]


class Course(BaseModel):
    course_name: str = Field(description="Short English course topic usable as a search query")
    target_skill: str = Field(description="The missing skill this course covers")
    level: Literal["beginner", "intermediate", "advanced"] = Field(description="Suitable level for the candidate")


class CourseRecommendations(BaseModel):
    courses: List[Course]


class ImprovedCV(BaseModel):
    improved_summary: str = Field(description="New professional summary, in the SAME language as the original CV")
    improved_bullet_points: List[str] = Field(
        description="Rewritten experience bullets (strong verb + task + result), same language as the "
                    "original CV, no invented numbers"
    )
    ats_tips: List[str] = Field(description="Tips to improve compatibility with applicant tracking systems (ATS)")


## 4) قراءة الملفات — `file_reader.py`
استخراج النص من PDF و Word (مع الجداول)، وتنظيفه، ورسائل خطأ واضحة.
يصلح النص العربي قبل إرساله للنموذج: الحروف المتصلة (ﺍﻟﺴﻴﺮﺓ)، والنص المعكوس (ةريسلا)، والتشكيل والرموز المخفية والأرقام العربية — لأن النص المشوّه أكبر سبب لهلوسة النموذج بالعربي.

In [ ]:
%%writefile file_reader.py
"""
file_reader.py — قراءة السيرة الذاتية من PDF أو Word وتنظيف النص.
"""
import io
import os
import re
import unicodedata
from typing import List, Tuple

from i18n import t

SUPPORTED_EXTENSIONS = (".pdf", ".docx")
MIN_CV_CHARS = 100      # أقل من هذا غالباً يعني PDF ممسوح ضوئياً
MAX_CV_CHARS = 15_000   # حد أعلى لطول النص المرسل للنموذج


class CVProcessingError(Exception):
    """خطأ مفهوم للمستخدم. code يطابق مفتاحاً في i18n: err_unsupported / err_corrupt / err_scanned."""

    def __init__(self, code: str, detail: str = ""):
        super().__init__(code)
        self.code = code
        self.detail = detail

    def message(self, lang: str) -> str:
        return str(t(lang, self.code))


# ---------- تنظيف النص العربي (أهم سبب للهلوسة: نص عربي مشوّه يصل للنموذج فيخمّن) ----------
_AR = r"\u0600-\u06FF"
_INVISIBLE = re.compile("[\u200b-\u200f\u202a-\u202e\u2066-\u2069\ufeff\u00ad]")
_TASHKEEL = re.compile("[\u064b-\u065f\u0670\u06d6-\u06ed]")
_LATIN_RUN = re.compile(r"[A-Za-z0-9@._+\-/:#%&()]+(?: +[A-Za-z0-9@._+\-/:#%&()]+)*")
_AR_WORD = re.compile(f"[{_AR}]{{2,}}")
_AR_ANY = re.compile("[\u0600-\u06FF\ufb50-\ufdff\ufe70-\ufeff]")
_AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")


def _fix_presentation_forms(text: str) -> str:
    """PDF أحياناً يخزن الحروف بأشكالها المتصلة (ﺍﻟﺴﻴﺮﺓ) فيراها النموذج رموزاً غريبة: نعيدها لحروف عادية."""
    return "".join(unicodedata.normalize("NFKC", ch) if "\ufb50" <= ch <= "\ufdff" or "\ufe70" <= ch <= "\ufeff"
                   else ch for ch in text)


def _is_reversed_arabic(text: str) -> bool:
    """بعض ملفات PDF تخرج العربي معكوساً (ةريسلا بدل السيرة). نكشفه من «ال» التعريف: في النص المعكوس تأتي آخر الكلمة."""
    words = _AR_WORD.findall(text)
    starts = sum(w.startswith("ال") for w in words)
    ends = sum(w.endswith("لا") for w in words)
    return ends >= 3 and ends > 2 * max(starts, 1)


def _reverse_line(line: str) -> str:
    """يعكس السطر العربي ثم يعيد الكلمات الإنجليزية والأرقام لاتجاهها الصحيح."""
    if not _AR_ANY.search(line):
        return line
    return _LATIN_RUN.sub(lambda m: m.group(0)[::-1], line[::-1])


def normalize_arabic(text: str) -> str:
    text = _INVISIBLE.sub("", text)
    # الكشف عن النص المعكوس يتم على نسخة مفكوكة الحروف، لكن العكس نفسه يتم قبل فك الحروف المتصلة:
    # حرف «لا» المركّب (ﻻ) رمز واحد، لو فككناه قبل العكس لصار «ال» (سلامه ← سالمه).
    if _is_reversed_arabic(_fix_presentation_forms(text)):
        text = "\n".join(_reverse_line(line) for line in text.split("\n"))
    text = _fix_presentation_forms(text)
    text = _TASHKEEL.sub("", text).replace("ـ", "")   # التشكيل والتطويل
    return text.translate(_AR_DIGITS)                     # ٢٠٢٤ → 2024 حتى يطابق النموذج الأرقام


def clean_text(text: str) -> str:
    """ينظف المسافات الزائدة مع الحفاظ على فواصل الأسطر (مهمة لفهم أقسام السيرة)، ويصلح النص العربي."""
    text = text.replace("\x00", "")
    text = normalize_arabic(text)
    text = re.sub(r"[ \t\u00a0]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_text_from_pdf(data: bytes) -> str:
    import fitz  # PyMuPDF

    with fitz.open(stream=data, filetype="pdf") as doc:
        return clean_text("\n".join(page.get_text("text", sort=True) for page in doc))  # sort: ترتيب القراءة الصحيح


def extract_text_from_docx(data: bytes) -> str:
    import docx

    document = docx.Document(io.BytesIO(data))
    parts = [p.text for p in document.paragraphs]
    for table in document.tables:  # كثير من السير مصممة بجداول
        for row in table.rows:
            cells: List[str] = []
            for cell in row.cells:
                txt = cell.text.strip()
                if txt and txt not in cells:  # الخلايا المدمجة تتكرر
                    cells.append(txt)
            if cells:
                parts.append(" | ".join(cells))
    return clean_text("\n".join(parts))


def extract_text(data: bytes, file_name: str) -> str:
    ext = os.path.splitext(file_name)[1].lower()
    if ext not in SUPPORTED_EXTENSIONS:
        raise CVProcessingError("err_unsupported")
    try:
        text = extract_text_from_pdf(data) if ext == ".pdf" else extract_text_from_docx(data)
    except Exception as e:
        raise CVProcessingError("err_corrupt", str(e)) from e
    if len(text) < MIN_CV_CHARS:
        raise CVProcessingError("err_scanned")
    return text


def extract_text_from_path(path: str) -> str:
    with open(path, "rb") as f:
        return extract_text(f.read(), path)


def prepare_cv_text(text: str) -> Tuple[str, bool]:
    """يقص النص الطويل جداً ويعيد (النص، هل تم القص)."""
    if len(text) > MAX_CV_CHARS:
        return text[:MAX_CV_CHARS], True
    return text, False


## 5) الوكلاء — `agents.py`
النموذج اللغوي والوكلاء الخمسة: استخراج البيانات، الفجوة المهارية، الوظائف، الدورات، تحسين السيرة.
كل وكيل عليه قواعد ضد الهلوسة: لا يستخدم إلا ما في السيرة، يكتب N/A لما هو غير مذكور، لا يترجم الأسماء، وعربي فصيح بدون كلمات مخترعة.

In [ ]:
%%writefile agents.py
"""
agents.py — النموذج اللغوي والوكلاء الخمسة (Prompt Templates + LCEL Chains).
"""
import os
from typing import Dict, Optional

from schemas import CandidateInfo, CourseRecommendations, ImprovedCV, JobRecommendations, SkillGap

DEFAULT_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

LANG_RULE = {
    "ar": "Write every descriptive field (summaries, reasons, tips, local titles) in correct, simple Modern Standard "
          "Arabic (فصحى) — no dialect, no invented or misspelled words, no literal word-by-word translation. "
          "Keep technical terms, tools, frameworks and certifications in English (Latin script). "
          "Never translate or transliterate names of people, companies or universities: copy them exactly as written.",
    "en": "Write every descriptive field (summaries, reasons, tips, local titles) in clear, professional English. "
          "Copy names of people, companies and universities exactly as written in the CV.",
}

# قواعد ضد الهلوسة تُضاف لكل الوكلاء
GROUNDING = (
    " STRICT RULES: Use ONLY facts stated in the provided text; never guess. "
    "If something is not stated (name, degree, dates, company), write 'N/A'. "
    "Never invent companies, employers, degrees, dates, numbers, certificates or achievements. "
    "The CV may be in Arabic and may contain extraction noise; if a passage is unreadable, ignore it instead of "
    "guessing what it says."
)

# كل وكيل: (تعليمات النظام، رسالة المستخدم)
PROMPTS = {
    "candidate": (
        "You are an expert HR analyst. Analyze the CV and extract the requested data precisely. "
        "Copy the candidate's full name exactly as written in the CV, word for word and in the same order. "
        "extracted_skills must contain only skills explicitly written in the CV. Compute years of experience from "
        "actual employment dates written in the CV, excluding years of study; if no dates are given, use 0. "
        "The summary must only restate what the CV says. {lang_rule}",
        "CV text:\n<cv>\n{cv_text}\n</cv>",
    ),
    "gap": (
        "You are a career advisor specialized in the Saudi labor market. Based on the candidate's field, "
        "title, experience and current skills, identify the in-demand skills they lack, ordered by priority. "
        "Never list a skill they already have. Base every reason on the candidate's stated field and title. {lang_rule}",
        "Field: {field}\nCurrent title: {title}\nYears of experience: {experience}\nCurrent skills: {skills}",
    ),
    "jobs": (
        "You are a recruitment expert in the Saudi market. Suggest 4 realistic job titles that fit the "
        "candidate's experience level and skills for work in {city}. Match percentages must be realistic and "
        "reflect missing skills. job_title must be the common English title used in LinkedIn job ads so it "
        "works as a search keyword, and job_title_local must be its accurate, commonly used translation. "
        "example_employers must be sectors or employer types (e.g. banks, government entities), not specific "
        "company names. {lang_rule}",
        "Current title: {title}\nField: {field}\nSkills: {skills}\nYears of experience: {experience}\nLocation: {city}",
    ),
    "courses": (
        "You are an academic advisor. Suggest one course for each high-priority missing skill (maximum 4 courses). "
        "course_name must be a short English phrase that works as a search query on Coursera and Udemy. {lang_rule}",
        "Candidate field: {field}\nYears of experience: {experience}\nMissing skills by priority: {missing_skills}",
    ),
    "improve": (
        "You are a professional CV writer. Rewrite the personal summary and experience bullet points to be "
        "professional and achievement-oriented (strong verb + task + result). Write improved_summary and "
        "improved_bullet_points in the SAME language as the original CV. Never invent numbers or achievements; "
        "when a number is needed, write [X] for the candidate to fill in. For ats_tips only: {lang_rule}",
        "Original CV text:\n<cv>\n{cv_text}\n</cv>",
    ),
}

SCHEMAS = {
    "candidate": CandidateInfo,
    "gap": SkillGap,
    "jobs": JobRecommendations,
    "courses": CourseRecommendations,
    "improve": ImprovedCV,
}


def build_llm(api_key: Optional[str] = None, model: Optional[str] = None):
    from langchain_openai import ChatOpenAI

    return ChatOpenAI(
        model=model or DEFAULT_MODEL,
        temperature=0,
        api_key=api_key or os.environ.get("OPENAI_API_KEY"),
        seed=7,          # نتائج ثابتة قدر الإمكان لنفس السيرة
        timeout=90,
        max_retries=3,
    )


def build_chains(llm, lang: str = "ar") -> Dict[str, object]:
    """يبني الوكلاء الخمسة: Prompt | LLM مع مخرجات منظمة (Pydantic)."""
    from langchain_core.prompts import ChatPromptTemplate

    rule = LANG_RULE.get(lang, LANG_RULE["ar"])
    return {
        name: ChatPromptTemplate.from_messages([("system", system.replace("{lang_rule}", rule)), ("human", human)])
        | llm.with_structured_output(SCHEMAS[name])
        for name, (system, human) in ((n, (sys_ + GROUNDING, hum)) for n, (sys_, hum) in PROMPTS.items())
    }


## 6) الروابط والتقرير — `report.py`
روابط بحث جاهزة للوظائف والدورات، وتقرير Markdown باللغة المختارة.

In [ ]:
%%writefile report.py
"""
report.py — روابط البحث عن الوظائف والدورات، وتقرير Markdown باللغة المختارة.
"""
from typing import Dict
from urllib.parse import urlencode

from file_reader import MAX_CV_CHARS
from i18n import CITIES, city_name, city_search, t


def _google_site_search(site: str, title: str, location: str) -> str:
    return "https://www.google.com/search?" + urlencode({"q": f'site:{site} "{title}" {location}'})


def job_links(job_title: str, city: str, lang: str = "ar") -> Dict[str, str]:
    city_en = city_search(city) if city in CITIES else None
    location = f"{city_en}, Saudi Arabia" if city_en else "Saudi Arabia"
    return {
        "LinkedIn": "https://www.linkedin.com/jobs/search/?" + urlencode({"keywords": job_title, "location": location}),
        "Indeed": "https://sa.indeed.com/jobs?" + urlencode({"q": job_title, "l": city_en or ""}),
        "Bayt": _google_site_search("bayt.com", job_title, city_en or "Saudi Arabia"),
        "Tanqeeb": _google_site_search("saudi.tanqeeb.com", job_title, city_en or "Saudi Arabia"),
        str(t(lang, "jadarat")): "https://jadarat.sa/",
    }


def course_links(course_name: str) -> Dict[str, str]:
    return {
        "Coursera": "https://www.coursera.org/search?" + urlencode({"query": course_name}),
        "Udemy": "https://www.udemy.com/courses/search/?" + urlencode({"q": course_name}),
        "LinkedIn Learning": "https://www.linkedin.com/learning/search?" + urlencode({"keywords": course_name}),
    }


def build_markdown_report(r) -> str:
    """r: AnalysisResult من pipeline.py"""
    L = lambda key, **kw: str(t(r.lang, key, **kw))  # noqa: E731
    prio, level = t(r.lang, "priority"), t(r.lang, "level")
    c, g = r.candidate, r.skill_gap
    lines = [
        f"# 📋 {L('r_title')}",
        "",
        f"## 👤 {L('r_info')}",
        f"- **{L('m_name')}:** {c.name}",
        f"- **{L('current_title')}:** {c.current_title}",
        f"- **{L('field')}:** {c.field}",
        f"- **{L('education')}:** {c.education}",
        f"- **{L('m_years')}:** {c.experience_years:g}",
        f"- **{L('skills')}:** {', '.join(c.extracted_skills)}",
        "",
        f"> {c.summary}",
        "",
        f"## 🔍 {L('gap_heading')}",
    ]
    lines += [f"- **{s.skill}** — {prio[s.priority]}: {s.reason}" for s in g.missing_skills]
    lines += ["", f"**{L('tips_heading')}:**"] + [f"- {tip}" for tip in g.recommendations_to_improve]

    lines += ["", f"## 💼 {L('jobs_heading', city=city_name(r.city, r.lang))}", f"_{L('jobs_note')}_", ""]
    for j in r.jobs.jobs:
        links = " · ".join(f"[{k}]({v})" for k, v in job_links(j.job_title, r.city, r.lang).items())
        title = j.job_title if r.lang == "en" else f"{j.job_title} — {j.job_title_local}"
        lines += [
            f"### {title} ({j.match_percentage}% {L('match')})",
            j.why_fit,
            f"- {L('employers')} {', '.join(j.example_employers)}",
            f"- {L('r_search')} {links}",
            "",
        ]

    lines += [f"## 📚 {L('courses_heading')}"]
    for co in r.courses.courses:
        links = " · ".join(f"[{k}]({v})" for k, v in course_links(co.course_name).items())
        lines.append(f"- **{co.course_name}** ({level[co.level]}) — {L('covers')} {co.target_skill} — {links}")

    imp = r.improved_cv
    lines += ["", f"## ✨ {L('summary_label')}", "", imp.improved_summary, ""]
    lines += [f"**{L('bullets_heading')}:**"] + [f"- {p}" for p in imp.improved_bullet_points]
    lines += ["", f"**{L('ats_heading')}:**"] + [f"- {tip}" for tip in imp.ats_tips]

    if r.truncated:
        lines += ["", f"_{L('r_truncated', n=f'{MAX_CV_CHARS:,}')}_"]
    return "\n".join(lines)


## 7) سير العمل — `pipeline.py`
تشغيل الوكلاء على ثلاث مراحل متوازية بدل خمس خطوات متتالية لتسريع التحليل.
بعد النموذج يتحقق `ground_results` من المخرجات: يحذف المهارات والاسم غير الموجودين في السيرة، ويحذف من الفجوة أي مهارة يملكها المرشح، ويستبدل أي رقم مخترع في السيرة المحسنة بـ [X].

In [ ]:
%%writefile pipeline.py
"""
pipeline.py — سير العمل: تشغيل الوكلاء الخمسة على ثلاث مراحل متوازية.
"""
import re
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from typing import Callable, List, Optional

import agents
from file_reader import prepare_cv_text
from i18n import CITIES, city_search, t
from schemas import CandidateInfo, CourseRecommendations, ImprovedCV, JobRecommendations, SkillGap


# ---------- تحقق بعد النموذج: نحذف ما لا يسنده نص السيرة (ضد الهلوسة) ----------
def _norm(text: str) -> str:
    """توحيد للمقارنة: حروف صغيرة، أ/إ/آ ← ا، ة ← ه، ى ← ي، بدون رموز."""
    text = text.lower()
    text = re.sub("[أإآ]", "ا", text).replace("ة", "ه").replace("ى", "ي")
    return re.sub(r"[^\w\s+#.]", " ", text)


def _in_text(phrase: str, norm_cv: str) -> bool:
    """العبارة موجودة في السيرة، أو كل كلماتها المهمة (3 أحرف فأكثر) موجودة."""
    p = _norm(phrase).strip()
    if not p:
        return False
    if p in norm_cv:
        return True
    words = [w for w in p.split() if len(w) >= 3]
    return bool(words) and all(w in norm_cv for w in words)


def _numbers_to_placeholder(line: str, cv_numbers: set) -> str:
    """أي رقم في النقاط المحسنة غير موجود في السيرة الأصلية يُستبدل بـ [X]."""
    return re.sub(r"\d+(?:[.,]\d+)?%?", lambda m: m.group(0) if m.group(0).rstrip("%") in cv_numbers else "[X]", line)


_AR_CHARS = re.compile("[\u0600-\u06FF]")


def _is_arabic(text: str) -> bool:
    letters = re.findall(r"[^\W\d_]", text)
    return bool(letters) and len(_AR_CHARS.findall(text)) / len(letters) > 0.3


def _name_as_written(name: str, cv_text: str) -> str:
    """إذا كانت كلمات الاسم موجودة في سطر واحد من السيرة بترتيب مختلف، نأخذه بالترتيب المكتوب في السيرة
    (النموذج أحياناً يبدّل ترتيب الأسماء العربية: سلامه لمياء ← لمياء سلامه)."""
    words = [_norm(w).strip() for w in name.split() if _norm(w).strip()]
    if len(words) < 2:
        return name
    target = sorted(words)
    for line in cv_text.splitlines():
        tokens = line.split()
        norm_tokens = [_norm(t).strip() for t in tokens]
        for i in range(len(tokens) - len(words) + 1):
            if sorted(norm_tokens[i:i + len(words)]) == target:
                return " ".join(re.sub(r"[^\w\-']", "", t) or t for t in tokens[i:i + len(words)])
    return name


def ground_results(cv_text: str, candidate: CandidateInfo, gap: SkillGap, improved: ImprovedCV) -> List[str]:
    """يعدّل المخرجات في مكانها ويعيد قائمة بما حُذف (للمراجعة).
    نتحقق فقط مما كُتب بنفس لغة السيرة، أو بالإنجليزية (المصطلحات التقنية دائماً إنجليزية)؛
    الترجمة العربية لسيرة إنجليزية لا تُحذف لأنها لن تُوجد حرفياً في النص."""
    norm_cv = _norm(cv_text)
    cv_ar = _is_arabic(cv_text)
    removed: List[str] = []

    def checkable(value: str) -> bool:
        return not _is_arabic(value) or cv_ar

    kept = [s for s in candidate.extracted_skills if not checkable(s) or _in_text(s, norm_cv)]
    removed += [s for s in candidate.extracted_skills if s not in kept]
    candidate.extracted_skills = kept

    if candidate.name.strip().upper() != "N/A":
        candidate.name = _name_as_written(candidate.name, cv_text)
    if candidate.name.strip().upper() != "N/A" and checkable(candidate.name) and not _in_text(candidate.name, norm_cv):
        removed.append(candidate.name)
        candidate.name = "N/A"

    have = {_norm(s).strip() for s in candidate.extracted_skills}
    gap.missing_skills = [m for m in gap.missing_skills
                          if _norm(m.skill).strip() not in have and not _in_text(m.skill, norm_cv)]

    cv_numbers = {n.replace(",", ".") for n in re.findall(r"\d+(?:[.,]\d+)?", cv_text)} | \
                 set(re.findall(r"\d+(?:[.,]\d+)?", cv_text))
    improved.improved_bullet_points = [_numbers_to_placeholder(b, cv_numbers) for b in improved.improved_bullet_points]
    improved.improved_summary = _numbers_to_placeholder(improved.improved_summary, cv_numbers)
    return removed


@dataclass
class AnalysisResult:
    candidate: CandidateInfo
    skill_gap: SkillGap
    jobs: JobRecommendations
    courses: CourseRecommendations
    improved_cv: ImprovedCV
    city: str            # مفتاح المدينة في CITIES
    lang: str = "ar"
    truncated: bool = False


def analyze_cv(
    cv_text: str,
    city: str = "riyadh",
    lang: str = "ar",
    llm=None,
    on_step: Optional[Callable[[str], None]] = None,
) -> AnalysisResult:
    """
    المراحل:
      1) استخراج البيانات  ‖  تحسين السيرة
      2) الفجوة المهارية  ‖  الوظائف
      3) الدورات
    on_step تُستدعى دائماً من الخيط الرئيسي، فهي آمنة مع Streamlit.
    """
    if city not in CITIES:
        city = "riyadh"
    step = on_step or (lambda _msg: None)
    cv_text, truncated = prepare_cv_text(cv_text)
    chains = agents.build_chains(llm or agents.build_llm(), lang)
    city_en = city_search(city)
    city_prompt = f"{city_en}, Saudi Arabia" if city_en else "any city in Saudi Arabia"

    with ThreadPoolExecutor(max_workers=3) as pool:
        step(str(t(lang, "step_1")))
        f_improve = pool.submit(chains["improve"].invoke, {"cv_text": cv_text})
        candidate: CandidateInfo = chains["candidate"].invoke({"cv_text": cv_text})
        # نحذف المهارات والاسم غير الموجودين في السيرة قبل ما تعتمد عليها باقي المراحل
        ground_results(cv_text, candidate, SkillGap(missing_skills=[], recommendations_to_improve=[]),
                       ImprovedCV(improved_summary="", improved_bullet_points=[], ats_tips=[]))

        base = {
            "field": candidate.field,
            "title": candidate.current_title,
            "experience": candidate.experience_years,
            "skills": ", ".join(candidate.extracted_skills) or "N/A",
        }

        step(str(t(lang, "step_2")))
        f_gap = pool.submit(chains["gap"].invoke, base)
        f_jobs = pool.submit(chains["jobs"].invoke, {**base, "city": city_prompt})
        gap: SkillGap = f_gap.result()

        step(str(t(lang, "step_3")))
        missing = ", ".join(f"{s.skill} ({s.priority})" for s in gap.missing_skills) or "none"
        courses: CourseRecommendations = chains["courses"].invoke(
            {"field": candidate.field, "experience": candidate.experience_years, "missing_skills": missing}
        )

        jobs: JobRecommendations = f_jobs.result()
        improved: ImprovedCV = f_improve.result()

    ground_results(cv_text, candidate, gap, improved)
    order = {"high": 0, "medium": 1, "low": 2}
    gap.missing_skills.sort(key=lambda s: order[s.priority])
    jobs.jobs.sort(key=lambda j: j.match_percentage, reverse=True)
    step(str(t(lang, "done")))
    return AnalysisResult(candidate, gap, jobs, courses, improved, city, lang, truncated)


## 8) البيانات التجريبية — `sample_data.py`
سيرة ونتائج تجريبية باللغتين يستخدمها «فحص النظام» لاختبار سير العمل بدون مفتاح.

In [ ]:
%%writefile sample_data.py
"""
sample_data.py — سيرة ونتائج تجريبية باللغتين يستخدمها «فحص النظام»
لاختبار قراءة الملفات وسير العمل بدون مفتاح OpenAI وبدون إنترنت.
"""
from pipeline import AnalysisResult
from schemas import (CandidateInfo, Course, CourseRecommendations, ImprovedCV, JobMatch,
                     JobRecommendations, MissingSkill, SkillGap)

SAMPLE_CV_TEXT = """Data Analyst — 5 years of experience
Experience: Data Analyst at a telecom company (2021 - present), Junior BI Developer (2019 - 2021)
Skills: Python, SQL, Power BI, Excel, Tableau, Statistics, Data Cleaning, Communication
Education: BSc in Information Systems"""

_DATA = {
    "ar": dict(
        candidate=dict(name="مرشح تجريبي", current_title="محلل بيانات", field="تحليل البيانات",
                       education="بكالوريوس نظم المعلومات",
                       summary="محلل بيانات بخبرة خمس سنوات في قطاع الاتصالات، يبني لوحات المؤشرات ويحوّل البيانات إلى قرارات.",
                       skills=["Python", "SQL", "Power BI", "Excel", "Tableau", "الإحصاء", "تنظيف البيانات", "التواصل"],
                       years=5),
        gap=[("Apache Spark", "high", "مطلوبة في أغلب وظائف البيانات الضخمة بالمملكة."),
             ("Cloud (AWS / Azure)", "high", "الجهات تنقل بياناتها للسحابة بشكل متسارع."),
             ("Machine Learning", "medium", "تفتح لك مسار علم البيانات لاحقاً."),
             ("Data Storytelling", "low", "تحسّن عرض نتائجك للإدارة.")],
        tips=["ابدأ بمشروع عملي صغير على Spark باستخدام بيانات مفتوحة.",
              "احصل على شهادة سحابية أساسية مثل AWS Cloud Practitioner.",
              "انشر مشاريعك على GitHub وأضف روابطها في سيرتك."],
        jobs=[("Data Analyst", "محلل بيانات", 85, "خبرتك في Python وSQL وPower BI تغطي أغلب متطلبات هذه الوظيفة.",
               ["البنوك", "شركات الاتصالات", "الجهات الحكومية"]),
              ("Business Intelligence Analyst", "محلل ذكاء أعمال", 78, "بناء لوحات المؤشرات من أبرز نقاط قوتك.",
               ["التجزئة", "الاستشارات", "الطاقة"]),
              ("Junior Data Engineer", "مهندس بيانات مبتدئ", 62, "أساسك في SQL قوي، وتعلّم Spark والسحابة يرفع مطابقتك.",
               ["شركات التقنية", "الاتصالات"])],
        summary="محلل بيانات بخبرة 5 سنوات في قطاع الاتصالات، متخصص في تحويل البيانات إلى رؤى تدعم القرار عبر Python وSQL وPower BI.",
        bullets=["بنيت [X] لوحة مؤشرات في Power BI استخدمتها الإدارة لمتابعة الأداء أسبوعياً.",
                 "أتمتة تقارير شهرية باستخدام Python وفّرت [X] ساعة عمل شهرياً.",
                 "حسّنت جودة البيانات عبر قواعد تنظيف موحدة قللت الأخطاء بنسبة [X]%."],
        ats=["استخدم عناوين أقسام قياسية مثل: الخبرات، المهارات، التعليم.",
             "أضف الكلمات المفتاحية من إعلان الوظيفة كما هي.",
             "تجنب الجداول والصور داخل السيرة عند التقديم الإلكتروني."],
    ),
    "en": dict(
        candidate=dict(name="Demo Candidate", current_title="Data Analyst", field="Data analysis",
                       education="BSc in Information Systems",
                       summary="Data analyst with five years in telecom who builds KPI dashboards and turns data into decisions.",
                       skills=["Python", "SQL", "Power BI", "Excel", "Tableau", "Statistics", "Data Cleaning", "Communication"],
                       years=5),
        gap=[("Apache Spark", "high", "Required in most big-data roles in Saudi Arabia."),
             ("Cloud (AWS / Azure)", "high", "Employers are moving their data to the cloud fast."),
             ("Machine Learning", "medium", "Opens a path into data science later."),
             ("Data Storytelling", "low", "Helps you present findings to management.")],
        tips=["Start a small hands-on Spark project with open data.",
              "Earn an entry-level cloud certificate such as AWS Cloud Practitioner.",
              "Publish your projects on GitHub and link them in your CV."],
        jobs=[("Data Analyst", "Data Analyst", 85, "Your Python, SQL and Power BI experience covers most of this role's requirements.",
               ["Banks", "Telecom companies", "Government entities"]),
              ("Business Intelligence Analyst", "Business Intelligence Analyst", 78,
               "Building KPI dashboards is one of your strongest points.", ["Retail", "Consulting", "Energy"]),
              ("Junior Data Engineer", "Junior Data Engineer", 62,
               "Strong SQL foundation; Spark and cloud skills would raise your match.", ["Tech companies", "Telecom"])],
        summary="Data analyst with 5 years in telecom, turning data into decision-ready insights with Python, SQL and Power BI.",
        bullets=["Built [X] Power BI dashboards that leadership used to track performance weekly.",
                 "Automated monthly reports with Python, saving [X] hours of work per month.",
                 "Improved data quality with standard cleaning rules, cutting errors by [X]%."],
        ats=["Use standard section titles such as Experience, Skills and Education.",
             "Mirror keywords from the job ad exactly.",
             "Avoid tables and images in your CV for online applications."],
    ),
}

_COURSES = [("Apache Spark for Data Engineering", "Apache Spark", "beginner"),
            ("AWS Cloud Practitioner", "Cloud (AWS / Azure)", "beginner"),
            ("Machine Learning Specialization", "Machine Learning", "intermediate")]


def sample_objects(lang: str = "ar") -> dict:
    """يعيد كائنات Pydantic لكل وكيل (تُستخدم أيضاً كبديل للنموذج في فحص النظام)."""
    d = _DATA.get(lang, _DATA["ar"])
    c = d["candidate"]
    return {
        "candidate": CandidateInfo(name=c["name"], current_title=c["current_title"], field=c["field"],
                                   education=c["education"], summary=c["summary"],
                                   extracted_skills=c["skills"], experience_years=c["years"]),
        "gap": SkillGap(missing_skills=[MissingSkill(skill=s, priority=p, reason=r) for s, p, r in d["gap"]],
                        recommendations_to_improve=d["tips"]),
        "jobs": JobRecommendations(jobs=[JobMatch(job_title=t, job_title_local=lt, match_percentage=p, why_fit=w,
                                                  example_employers=e) for t, lt, p, w, e in d["jobs"]]),
        "courses": CourseRecommendations(courses=[Course(course_name=n, target_skill=s, level=lv)
                                                  for n, s, lv in _COURSES]),
        "improve": ImprovedCV(improved_summary=d["summary"], improved_bullet_points=d["bullets"], ats_tips=d["ats"]),
    }


def sample_result(lang: str = "ar", city: str = "riyadh") -> AnalysisResult:
    o = sample_objects(lang)
    return AnalysisResult(o["candidate"], o["gap"], o["jobs"], o["courses"], o["improve"], city, lang, False)


## 9) فحص النظام — `checks.py`
عشرة اختبارات تعمل من داخل الواجهة (صفحة «فحص النظام») وتعرض نتيجة كل جزء ومدته.

In [ ]:
%%writefile checks.py
"""
checks.py — فحص النظام: اختبارات تُشغَّل من داخل الواجهة (صفحة "فحص النظام").
كل اختبار يعيد تفاصيل نجاحه، أو يرمي استثناءً عند الفشل، أو SkipCheck عند التخطي.
"""
import io
import time
from dataclasses import dataclass
from typing import Callable, List, Optional, Tuple

import agents
import file_reader
import i18n
import pipeline
import report
import sample_data
import schemas


class SkipCheck(Exception):
    pass


@dataclass
class CheckResult:
    key: str
    status: str      # pass / fail / skip
    detail: str
    seconds: float


def _i18n(lang, **_):
    missing = set(i18n.TEXT["ar"]) ^ set(i18n.TEXT["en"])
    assert not missing, f"missing keys: {sorted(missing)}"
    return i18n.t(lang, "chk_d_i18n", n=len(i18n.TEXT["ar"]))


def _schemas(lang, **_):
    job = schemas.JobMatch(job_title="x", job_title_local="x", match_percentage=150, why_fit="x", example_employers=[])
    cand = schemas.CandidateInfo(name="x", current_title="x", field="x", education="x", summary="x",
                                 extracted_skills=[], experience_years=-2)
    assert job.match_percentage == 100 and cand.experience_years == 0
    return i18n.t(lang, "chk_d_schemas")


def _word(lang, **_):
    import docx
    d = docx.Document()
    d.add_paragraph(sample_data.SAMPLE_CV_TEXT)
    table = d.add_table(rows=1, cols=2)
    table.cell(0, 0).text, table.cell(0, 1).text = "Tableau", "Statistics"
    buf = io.BytesIO()
    d.save(buf)
    text = file_reader.extract_text(buf.getvalue(), "sample.DOCX")
    assert "Tableau | Statistics" in text
    return i18n.t(lang, "chk_d_word", n=len(text))


def _pdf(lang, **_):
    import fitz
    doc = fitz.open()
    page = doc.new_page()
    page.insert_text((72, 72), sample_data.SAMPLE_CV_TEXT.replace("—", "-"), fontsize=11)
    data = doc.tobytes()
    doc.close()
    text = file_reader.extract_text(data, "sample.pdf")
    assert "Power BI" in text
    return i18n.t(lang, "chk_d_pdf", n=len(text))


def _unsupported(lang, **_):
    try:
        file_reader.extract_text(b"...", "cv.txt")
    except file_reader.CVProcessingError as e:
        assert e.code == "err_unsupported"
        return i18n.t(lang, "chk_d_unsupported")
    raise AssertionError("no error raised")


def _links(lang, **_):
    links = {**report.job_links("Data Analyst", "riyadh", lang), **report.course_links("Apache Spark")}
    assert all(u.startswith("https://") for u in links.values())
    return i18n.t(lang, "chk_d_links", n=len(links))


def _agents(lang, api_key=None, model=None, **_):
    chains = agents.build_chains(agents.build_llm(api_key=api_key or "sk-test", model=model), lang)
    assert len(chains) == 5
    return i18n.t(lang, "chk_d_agents", n=len(chains))


def _pipeline(lang, **_):
    """يشغّل سير العمل كاملاً بنموذج تجريبي (بدون إنترنت) للتأكد من ترابط المراحل."""
    fake = sample_data.sample_objects(lang)

    class FakeChain:
        def __init__(self, name):
            self.name = name

        def invoke(self, _inputs):
            return fake[self.name]

    original = agents.build_chains
    agents.build_chains = lambda _llm, _lang: {k: FakeChain(k) for k in fake}
    try:
        result = pipeline.analyze_cv(sample_data.SAMPLE_CV_TEXT, lang=lang, llm=object())
    finally:
        agents.build_chains = original
    assert result.jobs.jobs[0].match_percentage >= result.jobs.jobs[-1].match_percentage
    assert report.build_markdown_report(result)
    return i18n.t(lang, "chk_d_pipeline")


def _arabic(lang, **_):
    """يتأكد أن النص العربي المشوّه يُصلح قبل وصوله للنموذج، وأن المخرجات غير المسنودة تُحذف."""
    logical = ["السيرة الذاتية", "محلل بيانات في شركة ACME منذ 2021", "المهارات: Python و Power BI والتحليل"]
    run = file_reader._LATIN_RUN
    visual = "\n".join(run.sub(lambda m: m.group(0)[::-1], line[::-1]) for line in logical)
    assert file_reader.clean_text(visual) == "\n".join(logical), "reversed Arabic not repaired"
    assert file_reader.clean_text("ﺍﻟﺴﻴﺮﺓ ﺍﻟﺬﺍﺗﻴﺔ") == "السيرة الذاتية", "presentation forms not repaired"
    name_line = "الاسم: لمياء سلامه الرشيدي"  # «لا» المركّبة يجب أن تبقى «لا» بعد العكس
    lig = run.sub(lambda m: m.group(0)[::-1], name_line.replace("لا", "\ufefb")[::-1])
    assert file_reader.clean_text(lig + "\n" + visual) .split("\n")[0] == name_line, "lam-alef broken"
    cand = schemas.CandidateInfo(name="أحمد علي", current_title="x", field="x", education="x", summary="x",
                                 extracted_skills=["Python", "Power BI", "Kubernetes"], experience_years=3)
    gap = schemas.SkillGap(missing_skills=[schemas.MissingSkill(skill="Python", priority="high", reason="x"),
                                           schemas.MissingSkill(skill="Apache Spark", priority="high", reason="x")],
                           recommendations_to_improve=[])
    imp = schemas.ImprovedCV(improved_summary="x", improved_bullet_points=["رفعت الدقة 40% منذ 2021"], ats_tips=[])
    pipeline.ground_results("\n".join(logical), cand, gap, imp)
    assert cand.extracted_skills == ["Python", "Power BI"] and cand.name == "N/A"
    named = schemas.CandidateInfo(name="سلامه لمياء الرشيدي", current_title="x", field="x", education="x",
                                  summary="x", extracted_skills=[], experience_years=0)
    pipeline.ground_results(name_line, named, schemas.SkillGap(missing_skills=[], recommendations_to_improve=[]),
                            schemas.ImprovedCV(improved_summary="", improved_bullet_points=[], ats_tips=[]))
    assert named.name == "لمياء سلامه الرشيدي", "name order not fixed"
    assert [m.skill for m in gap.missing_skills] == ["Apache Spark"]
    assert imp.improved_bullet_points == ["رفعت الدقة [X] منذ 2021"]
    return i18n.t(lang, "chk_d_arabic")


def _openai(lang, api_key=None, model=None, **_):
    if not api_key:
        raise SkipCheck(i18n.t(lang, "chk_d_openai_skip"))
    reply = agents.build_llm(api_key=api_key, model=model).invoke("Reply with one word: ok").content
    return i18n.t(lang, "chk_d_openai", model=model or agents.DEFAULT_MODEL, reply=reply.strip()[:20])


CHECKS: List[Tuple[str, Callable]] = [
    ("i18n", _i18n), ("schemas", _schemas), ("word", _word), ("pdf", _pdf),
    ("unsupported", _unsupported), ("links", _links), ("agents", _agents),
    ("pipeline", _pipeline), ("arabic", _arabic), ("openai", _openai),
]


def run_all(lang: str = "ar", api_key: Optional[str] = None, model: Optional[str] = None,
            on_result: Optional[Callable[[CheckResult], None]] = None) -> List[CheckResult]:
    results = []
    for key, fn in CHECKS:
        start = time.perf_counter()
        try:
            status, detail = "pass", str(fn(lang, api_key=api_key, model=model))
        except SkipCheck as e:
            status, detail = "skip", str(e)
        except Exception as e:  # noqa: BLE001 — نعرض أي فشل للمستخدم
            status, detail = "fail", f"{type(e).__name__}: {e}"
        res = CheckResult(key, status, detail, time.perf_counter() - start)
        results.append(res)
        if on_result:
            on_result(res)
    return results


## 10) شعار الجامعة — `assets.py`
شعار الكلية التطبيقية بجامعة الأمير مساعد بن عبدالرحمن (بدون خلفية) محفوظ داخل الكود بصيغة base64، فيظهر في الواجهة بدون رفع أي ملف.

In [ ]:
%%writefile assets.py
"""
assets.py — شعار الكلية التطبيقية بجامعة الأمير مساعد بن عبدالرحمن بصيغة base64 (بدون خلفية) حتى يعمل بدون ملفات خارجية.
LOGO: الألوان الأصلية للخلفيات الفاتحة · LOGO_WHITE: نص أبيض للخلفيات الكحلية · LOGO_MARK: رمز الشعار فقط بدون كتابة.
"""
LOGO = "..."
LOGO_WHITE = "..."
LOGO_MARK = "..."


## 11) عدّاد الزوار — `visitors.py`
يحفظ عدد الزوار في `visitors.json` ويزيده مرة واحدة لكل جلسة، مع عدد زوار اليوم. يظهر بالإنجليزية في التذييل.

In [ ]:
%%writefile visitors.py
"""
visitors.py — عدّاد الزوار: يحفظ العدد في ملف JSON ويزيده مرة واحدة لكل جلسة.
القفل (Lock) يمنع تلف الملف إذا دخل أكثر من زائر في نفس اللحظة.
"""
import json
import os
import threading
from datetime import date

VISITORS_PATH = os.environ.get("CV_AGENT_VISITORS", "visitors.json")
_LOCK = threading.Lock()


def _read() -> dict:
    try:
        with open(VISITORS_PATH, encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def register_visit() -> dict:
    """يزيد العداد الكلي وعداد اليوم، ويعيد {"total", "today"}."""
    today = date.today().isoformat()
    with _LOCK:
        data = _read()
        data["total"] = int(data.get("total", 0)) + 1
        days = data.get("days", {})
        days[today] = int(days.get(today, 0)) + 1
        data["days"] = dict(sorted(days.items())[-60:])  # آخر 60 يوماً فقط
        tmp = VISITORS_PATH + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(data, f)
        os.replace(tmp, VISITORS_PATH)  # كتابة آمنة
    return {"total": data["total"], "today": days[today]}


def get_counts() -> dict:
    data = _read()
    return {"total": int(data.get("total", 0)),
            "today": int(data.get("days", {}).get(date.today().isoformat(), 0))}


## 12) واجهة المستخدم — `app.py`
صفحتان: **التحليل** و **فحص النظام**، مع رمز شعار الجامعة في الشريط العلوي والشعار كاملاً في التذييل، بتصميم Figma المبني على قالب Simple Design System باللونين الأزرق الغامق والفاتح.

In [ ]:
%%writefile app.py
"""
app.py — واجهة Streamlit لوكيل السيرة الذاتية (تصميم Figma مبني على قالب Simple Design System)
الصفحات: التحليل · فحص النظام        التشغيل:  streamlit run app.py
"""
import hashlib
import html
import os

import streamlit as st

import checks
from assets import LOGO_MARK, LOGO_WHITE
import visitors
from agents import DEFAULT_MODEL, build_llm
from file_reader import CVProcessingError, extract_text
from i18n import CITIES, city_name, t
from pipeline import analyze_cv
from report import build_markdown_report, course_links, job_links

# ================= الحالة العامة =================
for key, default in {"lang": "ar", "page": 0}.items():
    st.session_state.setdefault(key, default)
LANG = st.session_state.lang
RTL = LANG == "ar"


def T(key, **kw):
    return t(LANG, key, **kw)


def esc(value) -> str:
    return html.escape(str(value))


def html_block(markup: str):
    st.markdown(markup, unsafe_allow_html=True)


st.set_page_config(page_title=T("page_title"), page_icon="📄", layout="wide", initial_sidebar_state="collapsed")

# ================= التصميم =================
html_block("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Readex+Pro:wght@400;500;600;700&display=swap');
:root{--navy:#0B1B3F;--navy2:#13306B;--blue:#1D4ED8;--blue2:#2563EB;--sky:#38BDF8;--soft:#EFF6FF;
      --soft-line:#BFDBFE;--bg:#E8EEF7;--line:#E2E8F0;--ink:#0F172A;--muted:#64748B;--body:#334155;
      --shadow:0 4px 16px rgba(11,27,63,.06);}
.stApp{background:var(--bg);}
.stApp :is(p,li,h1,h2,h3,h4,label,button,input,textarea,div){font-family:'Readex Pro',sans-serif;}
[data-testid="stHeader"],[data-testid="stSidebar"],[data-testid="stSidebarCollapsedControl"]{display:none;}
.block-container{padding-top:1.2rem;max-width:1400px;}

/* شريط التنقل العلوي */
.st-key-topnav{background:linear-gradient(90deg,var(--navy),var(--navy2));border-radius:16px;padding:10px 20px;
      box-shadow:0 6px 20px rgba(11,27,63,.15);}
.cv-brand{display:flex;align-items:center;gap:12px;}
.cv-logo{width:40px;height:40px;border-radius:11px;background:linear-gradient(135deg,var(--blue2),var(--sky));
      display:flex;align-items:center;justify-content:center;font-weight:700;font-size:15px;color:#fff;flex-shrink:0;}
.cv-brand b{display:block;font-size:17px;color:#fff;line-height:1.2;}
.cv-brand small{font-size:11px;color:#BFDBFE;}
.st-key-topnav [data-testid="stButtonGroup"]{background:rgba(255,255,255,.07);border-radius:999px;padding:4px;
      width:fit-content;}
.st-key-topnav [data-testid="stButtonGroup"] button{background:transparent;border:none;color:#DBEAFE;
      border-radius:999px;padding:6px 18px;font-weight:500;white-space:nowrap;}
.st-key-topnav [data-testid="stButtonGroup"] button[kind*="Active"],
.st-key-topnav [data-testid="stButtonGroup"] button[aria-checked="true"]{background:#fff!important;
      color:var(--navy)!important;font-weight:600;}
.st-key-nav_lang [data-testid="stButtonGroup"] button[kind*="Active"],
.st-key-nav_lang [data-testid="stButtonGroup"] button[aria-checked="true"]{
      background:linear-gradient(90deg,var(--blue2),#3B82F6)!important;color:#fff!important;}
.st-key-nav_lang{display:flex;justify-content:flex-end;}

/* البانر */
.st-key-hero{position:relative;overflow:hidden;border-radius:22px;padding:36px 40px;margin-top:8px;
      background:linear-gradient(135deg,var(--navy2),var(--blue2));box-shadow:0 10px 30px rgba(11,27,63,.18);}
.st-key-hero::before{content:"";position:absolute;width:300px;height:300px;border-radius:50%;
      background:rgba(56,189,248,.12);inset-inline-end:-80px;top:-140px;}
.cv-tag-hero{display:inline-flex;align-items:center;gap:6px;padding:5px 12px;border-radius:999px;
      background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.2);color:#E0F2FE;font-size:12px;}
.cv-hero-title{font-size:34px;font-weight:700;color:#fff;margin:12px 0 8px;line-height:1.25;}
.cv-hero-sub{font-size:16px;color:#DBEAFE;margin:0 0 18px;}
.cv-btn-white{display:inline-block;background:#fff;color:var(--navy)!important;font-weight:600;padding:9px 22px;
      border-radius:12px;text-decoration:none!important;}
.cv-steps{background:rgba(255,255,255,.09);border:1px solid rgba(255,255,255,.18);border-radius:18px;padding:20px 22px;}
.cv-steps h4{color:#fff;font-size:15px;font-weight:600;margin:0 0 14px;padding:0;}
.cv-step{display:flex;gap:12px;align-items:flex-start;margin-bottom:12px;}
.cv-step i{width:28px;height:28px;border-radius:50%;background:#fff;color:var(--navy);font-style:normal;
      font-weight:700;font-size:13px;display:flex;align-items:center;justify-content:center;flex-shrink:0;}
.cv-step b{display:block;color:#fff;font-size:14px;}
.cv-step span{color:#BFDBFE;font-size:12px;}

/* لوحة التحكم والبطاقات */
.st-key-panel,.st-key-chk_header{background:#fff;border:1px solid var(--line);border-radius:16px;padding:22px;
      box-shadow:var(--shadow);margin-top:8px;}
[data-testid="stFileUploaderDropzone"]{background:rgba(239,246,255,.6);border:1.5px dashed rgba(37,99,235,.4);
      border-radius:14px;}
[data-testid="stFileUploaderDropzone"] svg{fill:var(--blue);}
.stTextInput input,[data-baseweb="select"]>div{border-radius:10px!important;}
.stButton button[kind="primary"],[data-testid="stBaseButton-primary"]{
      background:linear-gradient(90deg,var(--blue),#3B82F6)!important;border:none!important;border-radius:12px;
      font-weight:600;padding:10px 22px;color:#fff!important;box-shadow:0 6px 14px rgba(29,78,216,.3);}
.stButton button[kind="primary"]:hover{filter:brightness(1.08);}
.stDownloadButton button{width:100%;background:var(--navy)!important;color:#fff!important;border:none!important;
      border-radius:12px;font-weight:600;padding:12px 16px;}

/* التبويبات (نمط Tab من القالب) */
.stTabs [data-baseweb="tab-list"]{gap:6px;border-bottom:1px solid var(--line);}
.stTabs [data-baseweb="tab"]{font-size:15px;font-weight:500;color:var(--muted);padding:10px 14px;}
.stTabs [aria-selected="true"]{color:var(--blue)!important;font-weight:600;}
.stTabs [data-baseweb="tab-highlight"]{background:var(--blue);height:3px;}

.cv-field{font-size:14px;color:var(--ink);font-weight:500;margin-bottom:6px;}
.cv-card{background:#fff;border:1px solid var(--line);border-radius:16px;padding:22px;margin-bottom:16px;
      box-shadow:var(--shadow);}
.cv-grid{display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin:8px 0 20px;}
.cv-grid .cv-card{margin:0;}
.cv-metric{display:flex;gap:14px;align-items:center;padding:18px;}
.cv-ic{width:44px;height:44px;border-radius:12px;background:var(--soft);display:flex;align-items:center;
      justify-content:center;flex-shrink:0;color:var(--blue);}
.cv-ic.accent{background:linear-gradient(135deg,var(--blue),var(--sky));color:#fff;}
.cv-label{font-size:12px;color:var(--muted);margin-bottom:2px;}
.cv-value{font-size:22px;font-weight:700;color:var(--ink);line-height:1.2;}
.cv-value.accent{color:var(--blue);}
.cv-h2{font-size:20px;font-weight:600;color:var(--ink);margin:12px 0 2px;}
.cv-h3{font-size:17px;font-weight:600;color:var(--ink);margin:0;}
.cv-title-row{display:flex;align-items:center;gap:10px;margin-bottom:14px;}
.cv-title-row .cv-ic{width:34px;height:34px;border-radius:10px;}
.cv-note{font-size:13px;color:var(--muted);margin:0 0 14px;}
.cv-sub{font-size:13px;color:var(--muted);}
.cv-text{font-size:14px;color:var(--body);margin:12px 0 6px;line-height:1.6;}
.cv-top{display:flex;gap:16px;align-items:flex-start;justify-content:space-between;}
.cv-match{background:var(--soft);border:1px solid var(--soft-line);border-radius:12px;padding:8px 14px;
      text-align:center;flex-shrink:0;}
.cv-match b{display:block;font-size:20px;color:var(--blue);line-height:1.2;}
.cv-match span{font-size:11px;color:var(--blue);font-weight:500;}
.cv-progress{height:6px;background:var(--line);border-radius:4px;overflow:hidden;margin-top:12px;}
.cv-progress div{height:100%;border-radius:4px;background:linear-gradient(90deg,var(--blue),var(--sky));}
.cv-chips{display:flex;flex-wrap:wrap;gap:8px;margin-top:12px;}
.cv-chip{display:inline-flex;align-items:center;gap:6px;padding:7px 14px;border-radius:10px;
      background:var(--soft);border:1px solid var(--soft-line);color:var(--blue)!important;
      font-size:13px;font-weight:500;text-decoration:none!important;transition:all .15s;}
.cv-chip:hover{background:var(--blue);color:#fff!important;border-color:var(--blue);}
.cv-tag{display:inline-block;padding:4px 10px;border-radius:8px;background:var(--soft);color:var(--blue);
      font-size:13px;margin:0 0 6px 6px;}
.cv-gap-item{display:flex;gap:12px;align-items:flex-start;margin-bottom:14px;}
.cv-gap-item b{font-size:14px;color:var(--ink);display:block;}
/* وسوم القالب (Tag: Danger / Warning / Positive / Brand) */
.cv-badge{padding:4px 10px;border-radius:8px;font-size:12px;font-weight:600;white-space:nowrap;}
.cv-badge.high,.cv-badge.fail{background:#FEE9E7;color:#900B09;}
.cv-badge.medium,.cv-badge.skip{background:#FFF1C2;color:#522504;}
.cv-badge.low,.cv-badge.pass{background:#CFF7D3;color:#02542D;}
.cv-badge.level{background:var(--soft);color:var(--blue);}
.cv-list{margin:0;padding-inline-start:20px;color:var(--body);font-size:14px;line-height:1.7;}
.cv-info{display:flex;gap:12px;align-items:flex-start;background:var(--soft);border:1px solid var(--soft-line);
      border-radius:14px;padding:14px 18px;color:var(--body);font-size:13px;margin-top:16px;}

/* فحص النظام */
.cv-chk-title{font-size:24px;font-weight:700;color:var(--ink);margin:0;}
.cv-mini{background:var(--bg);border-radius:12px;padding:12px 16px;}
.cv-chk-bar{height:6px;border-radius:4px;background:linear-gradient(90deg,var(--blue),#22C55E);margin-top:16px;}
.cv-chk-row{display:flex;align-items:center;gap:16px;padding:16px 0;border-bottom:1px solid var(--line);}
.cv-chk-row:last-child{border-bottom:none;}
.cv-chk-row .st{width:36px;height:36px;border-radius:50%;display:flex;align-items:center;justify-content:center;
      flex-shrink:0;}
.cv-chk-row .st.pass{background:#DCFCE7;color:#15803D;}
.cv-chk-row .st.fail{background:#FEE2E2;color:#DC2626;}
.cv-chk-row .st.skip{background:#FEF3C7;color:#B45309;}
.cv-chk-row .tx{flex:1;min-width:0;}
.cv-chk-row .tx b{font-size:15px;color:var(--ink);display:block;}
.cv-chk-row .tm{font-size:12px;color:var(--muted);white-space:nowrap;}
@media (max-width:900px){.cv-grid{grid-template-columns:repeat(2,1fr);}}
/* ===== الحركة: خلفية متحركة + ظهور تدريجي + تفاعل ===== */
.stApp{background:linear-gradient(120deg,#E8EEF7 0%,#DCE7F7 35%,#E8EEF7 60%,#E3E9F8 100%);
      background-size:300% 300%;animation:cvBg 22s ease infinite;}
.stApp::before,.stApp::after{content:"";position:fixed;border-radius:50%;filter:blur(70px);z-index:0;
      pointer-events:none;opacity:.55;}
.stApp::before{width:520px;height:520px;background:radial-gradient(circle,#93C5FD,transparent 70%);
      top:-120px;inset-inline-start:-140px;animation:cvFloatA 26s ease-in-out infinite;}
.stApp::after{width:460px;height:460px;background:radial-gradient(circle,#BAE6FD,transparent 70%);
      bottom:-140px;inset-inline-end:-120px;animation:cvFloatB 30s ease-in-out infinite;}
[data-testid="stAppViewContainer"],[data-testid="stMain"]{background:transparent;position:relative;z-index:1;}
.st-key-hero{background-size:200% 200%!important;animation:cvBg 14s ease infinite,cvRise .6s ease both;}
.st-key-hero::before{animation:cvFloatA 18s ease-in-out infinite;}
.st-key-hero::after{content:"";position:absolute;width:180px;height:180px;border-radius:50%;
      background:rgba(56,189,248,.10);inset-inline-end:280px;bottom:-90px;animation:cvFloatB 16s ease-in-out infinite;}
.st-key-topnav{animation:cvDrop .5s ease both;}
.st-key-panel,.st-key-chk_header,.cv-card,.cv-mini,.cv-info{animation:cvRise .55s ease both;}
.cv-grid .cv-card:nth-child(2){animation-delay:.06s}.cv-grid .cv-card:nth-child(3){animation-delay:.12s}
.cv-grid .cv-card:nth-child(4){animation-delay:.18s}
.cv-card{transition:transform .2s ease,box-shadow .2s ease;}
.cv-card:hover{transform:translateY(-3px);box-shadow:0 12px 28px rgba(11,27,63,.10);}
.cv-progress div{animation:cvGrow 1.1s cubic-bezier(.2,.8,.2,1) both;}
.cv-chk-row{animation:cvRise .4s ease both;}
.cv-logo{animation:cvPulse 3.5s ease-in-out infinite;}
.stButton button[kind="primary"]{transition:transform .15s ease,filter .15s ease;}
.stButton button[kind="primary"]:hover{transform:translateY(-1px);}
@keyframes cvBg{0%{background-position:0% 50%}50%{background-position:100% 50%}100%{background-position:0% 50%}}
@keyframes cvFloatA{0%,100%{transform:translate(0,0)}50%{transform:translate(60px,40px)}}
@keyframes cvFloatB{0%,100%{transform:translate(0,0)}50%{transform:translate(-50px,-35px)}}
@keyframes cvRise{from{opacity:0;transform:translateY(12px)}to{opacity:1;transform:none}}
@keyframes cvDrop{from{opacity:0;transform:translateY(-10px)}to{opacity:1;transform:none}}
@keyframes cvGrow{from{width:0}}
@keyframes cvPulse{0%,100%{box-shadow:0 0 0 0 rgba(56,189,248,.45)}50%{box-shadow:0 0 0 8px rgba(56,189,248,0)}}
/* ===== لمسات احترافية: شعار الجامعة + تحسينات عامة ===== */
::selection{background:rgba(37,99,235,.18);}
::-webkit-scrollbar{width:10px;height:10px}::-webkit-scrollbar-thumb{background:#C7D4E8;border-radius:10px;
      border:2px solid #E8EEF7}::-webkit-scrollbar-track{background:transparent}
.st-key-topnav{position:sticky;top:10px;z-index:50;backdrop-filter:blur(10px);
      background:linear-gradient(90deg,rgba(11,27,63,.96),rgba(19,48,107,.96))!important;
      border:1px solid rgba(255,255,255,.08);padding:12px 22px!important;}
.cv-uni-logo{height:46px;width:auto;flex-shrink:0;filter:drop-shadow(0 2px 6px rgba(0,0,0,.25));}
.cv-sep{width:1px;height:34px;background:rgba(255,255,255,.18);flex-shrink:0;}
.cv-pmau{font-weight:700;font-size:13px;letter-spacing:1.5px;color:#7DD3FC;background:rgba(56,189,248,.12);
      border:1px solid rgba(125,211,252,.3);padding:4px 10px;border-radius:8px;flex-shrink:0;}
.cv-brand .cv-logo{width:36px;height:36px;font-size:13px;border-radius:10px;}
.cv-hero-tags{display:flex;flex-wrap:wrap;gap:8px;}
.cv-tag-hero.uni{background:rgba(11,27,63,.28);}
.cv-tag-hero.uni b{color:#7DD3FC;letter-spacing:1px;margin-inline-end:4px;}
.cv-hero-title{font-size:38px!important;letter-spacing:-.3px;}
.cv-hero-cta{display:flex;flex-wrap:wrap;align-items:center;gap:10px;}
.cv-btn-white{display:inline-flex!important;align-items:center;gap:8px;box-shadow:0 8px 20px rgba(0,0,0,.18);
      transition:transform .15s ease,box-shadow .15s ease;}
.cv-btn-white:hover{transform:translateY(-2px);box-shadow:0 12px 26px rgba(0,0,0,.24);}
.cv-stat{font-size:12.5px;color:#E0F2FE;padding:7px 12px;border-radius:999px;background:rgba(255,255,255,.08);
      border:1px solid rgba(255,255,255,.14);}
.st-key-panel,.st-key-chk_header{position:relative;overflow:hidden;border-radius:20px!important;}
.st-key-panel::before,.st-key-chk_header::before{content:"";position:absolute;top:0;left:0;right:0;height:4px;
      background:linear-gradient(90deg,var(--sky),var(--blue),var(--navy2));}
[data-testid="stFileUploaderDropzone"]{transition:border-color .15s ease,background .15s ease;}
[data-testid="stFileUploaderDropzone"]:hover{border-color:var(--blue)!important;background:#EFF6FF!important;}
.stTextInput input:focus,[data-baseweb="select"]>div:focus-within{border-color:var(--blue2)!important;
      box-shadow:0 0 0 3px rgba(37,99,235,.15)!important;}
.stTabs [data-baseweb="tab-list"]{background:#fff;border:1px solid var(--line);border-radius:14px;padding:6px;
      gap:4px;box-shadow:var(--shadow);}
.stTabs [data-baseweb="tab"]{border-radius:10px;padding:8px 16px;}
.stTabs [aria-selected="true"]{background:var(--soft);}
.stTabs [data-baseweb="tab-highlight"],.stTabs [data-baseweb="tab-border"]{display:none;}
.cv-grid .cv-card{position:relative;overflow:hidden;}
.cv-grid .cv-card::after{content:"";position:absolute;bottom:0;inset-inline-start:0;width:42%;height:3px;
      background:linear-gradient(90deg,var(--blue),var(--sky));border-radius:3px;}
.cv-card{border-radius:18px;}
.cv-footer{margin:34px 0 10px;padding:22px 26px;border-radius:20px;color:#DBEAFE;display:flex;flex-wrap:wrap;
      align-items:center;justify-content:space-between;gap:18px;
      background:linear-gradient(135deg,var(--navy2),var(--navy));box-shadow:0 10px 30px rgba(11,27,63,.18);
      animation:cvRise .6s ease both;}
.cv-f-main{display:flex;align-items:center;gap:16px;}
.cv-f-main img{height:72px;width:auto;}
.cv-f-main b{display:block;color:#fff;font-size:15px;}
.cv-f-main span{font-size:12.5px;color:#BFDBFE;}
.cv-f-meta{display:flex;flex-direction:column;gap:4px;font-size:12.5px;color:#BFDBFE;text-align:end;}
@media (max-width:900px){.cv-uni-logo{height:36px}.cv-hero-title{font-size:28px!important}
      .cv-f-meta{text-align:start}}
/* ===== البانر: تصميم جديد بألوان شعار الجامعة (كحلي + فيروزي + بنفسجي) ===== */
.st-key-hero{background:
      radial-gradient(520px 320px at 88% -10%,rgba(31,181,198,.38),transparent 70%),
      radial-gradient(460px 300px at 8% 115%,rgba(109,75,196,.42),transparent 70%),
      radial-gradient(rgba(255,255,255,.07) 1px,transparent 1.4px) 0 0/22px 22px,
      linear-gradient(135deg,#0B1B3F 0%,#102A5C 55%,#1A2468 100%)!important;
      background-size:auto,auto,22px 22px,auto!important;animation:cvRise .6s ease both!important;
      border:1px solid rgba(125,227,238,.18);box-shadow:0 18px 44px rgba(11,27,63,.28),inset 0 1px 0 rgba(255,255,255,.08);
      padding:42px 44px!important;}
.st-key-hero::before{width:240px!important;height:240px!important;top:-120px!important;
      background:conic-gradient(from 200deg,rgba(31,181,198,.0),rgba(31,181,198,.35),rgba(109,75,196,.35),rgba(31,181,198,0))!important;
      filter:blur(30px);}
.st-key-hero::after{background:rgba(109,75,196,.18)!important;filter:blur(20px);}
.cv-tag-hero{background:rgba(31,181,198,.14)!important;border:1px solid rgba(125,227,238,.32)!important;color:#CFFAFE!important;}
.cv-tag-hero.uni{background:rgba(109,75,196,.18)!important;border-color:rgba(167,139,250,.35)!important;color:#EDE9FE!important;}
.cv-hero-title{color:#fff!important;font-size:40px!important;text-shadow:0 4px 24px rgba(31,181,198,.25);}
.cv-hero-title::after{content:"";display:block;width:84px;height:4px;border-radius:4px;margin-top:12px;
      background:linear-gradient(90deg,#1FB5C6,#6D4BC4);}
.cv-hero-sub{color:#C7D2FE!important;max-width:640px;}
.cv-btn-white{background:linear-gradient(90deg,#14B8A6,#22D3EE)!important;color:#06263A!important;font-weight:700!important;
      box-shadow:0 10px 24px rgba(20,184,166,.35)!important;}
.cv-btn-white:hover{box-shadow:0 14px 30px rgba(20,184,166,.45)!important;}
.cv-stat{background:rgba(255,255,255,.06)!important;border:1px solid rgba(125,227,238,.25)!important;color:#E0F7FA!important;}
.cv-steps{position:relative;background:linear-gradient(160deg,rgba(255,255,255,.10),rgba(255,255,255,.03))!important;
      border:1px solid rgba(125,227,238,.22)!important;border-radius:20px!important;padding:22px 24px!important;
      backdrop-filter:blur(8px);box-shadow:0 12px 30px rgba(0,0,0,.18);}
.cv-steps h4{color:#7DE3EE!important;letter-spacing:.3px;}
.cv-step{position:relative;margin-bottom:16px!important;}
.cv-step:not(:last-child)::after{content:"";position:absolute;inset-inline-start:14px;top:32px;width:2px;height:calc(100% - 14px);
      background:linear-gradient(180deg,rgba(31,181,198,.6),rgba(109,75,196,.5));}
.cv-step i{background:linear-gradient(135deg,#1FB5C6,#6D4BC4)!important;color:#fff!important;
      box-shadow:0 0 0 4px rgba(31,181,198,.15);}
.cv-step span{color:#C7D2FE!important;}
/* ===== عدّاد الزوار ===== */
.cv-visits{display:inline-flex;align-items:center;gap:7px;align-self:flex-end;font-size:12.5px;color:#E0F7FA;
      background:rgba(31,181,198,.14);border:1px solid rgba(125,227,238,.3);padding:5px 12px;border-radius:999px;
      font-family:'Readex Pro',sans-serif;}
.cv-visits b{color:#fff;font-weight:700;}
.cv-visits span{opacity:.5;}
.cv-visits i{width:7px;height:7px;border-radius:50%;background:#22D3EE;box-shadow:0 0 0 3px rgba(34,211,238,.25);
      animation:dmBlinkV 1.6s ease-in-out infinite;}
@keyframes dmBlinkV{50%{opacity:.35}}
.st-key-topnav .cv-visits{align-self:auto;white-space:nowrap;font-size:11.5px;padding:5px 10px;}
@media (prefers-reduced-motion:reduce){*,*::before,*::after{animation:none!important;transition:none!important;}}
</style>
""")
if RTL:
    html_block("""<style>
[data-testid="stMain"],.main{direction:rtl;text-align:right;}
.cv-tag{margin:0 6px 6px 0;}
.st-key-nav_lang{justify-content:flex-start;}
</style>""")


# ================= أيقونات =================
def svg(*paths, size=20) -> str:
    return (f'<svg width="{size}" height="{size}" viewBox="0 0 24 24" fill="none" stroke="currentColor" '
            'stroke-width="2" stroke-linecap="round" stroke-linejoin="round">'
            + "".join(f'<path d="{p}"/>' for p in paths) + "</svg>")


P = {
    "user": ("M19 21v-2a4 4 0 0 0-4-4H9a4 4 0 0 0-4 4v2", "M12 11a4 4 0 1 0 0-8 4 4 0 0 0 0 8z"),
    "brief": ("M16 20V4a2 2 0 0 0-2-2h-4a2 2 0 0 0-2 2v16",
              "M20 6H4a2 2 0 0 0-2 2v10a2 2 0 0 0 2 2h16a2 2 0 0 0 2-2V8a2 2 0 0 0-2-2z"),
    "list": ("M8 6h13", "M8 12h13", "M8 18h13", "M3 6h.01", "M3 12h.01", "M3 18h.01"),
    "target": ("M12 22a10 10 0 1 0 0-20 10 10 0 0 0 0 20z", "M12 18a6 6 0 1 0 0-12 6 6 0 0 0 0 12z",
               "M12 14a2 2 0 1 0 0-4 2 2 0 0 0 0 4z"),
    "spark": ("M12 3l1.9 5.8L20 10l-6.1 1.2L12 17l-1.9-5.8L4 10l6.1-1.2z",),
    "book": ("M4 19.5A2.5 2.5 0 0 1 6.5 17H20", "M6.5 2H20v20H6.5A2.5 2.5 0 0 1 4 19.5v-15A2.5 2.5 0 0 1 6.5 2z"),
    "upload": ("M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4", "M17 8l-5-5-5 5", "M12 3v12"),
    "check": ("M20 6 9 17l-5-5",),
    "x": ("M18 6 6 18", "M6 6l12 12"),
    "minus": ("M5 12h14",),
    "clock": ("M12 22a10 10 0 1 0 0-20 10 10 0 0 0 0 20z", "M12 6v6l4 2"),
    "info": ("M12 22a10 10 0 1 0 0-20 10 10 0 0 0 0 20z", "M12 16v-4", "M12 8h.01"),
    "arrow": ("M7 17 17 7", "M7 7h10v10"),
}


def icon(name, size=20):
    return svg(*P[name], size=size)


def title_row(text: str, ic: str) -> str:
    return f'<div class="cv-title-row"><div class="cv-ic">{icon(ic, 18)}</div><div class="cv-h3">{esc(text)}</div></div>'


def field_label(text: str):
    html_block(f'<div class="cv-field">{esc(text)}</div>')


def chips(links: dict) -> str:
    return '<div class="cv-chips">' + "".join(
        f'<a class="cv-chip" href="{esc(u)}" target="_blank" rel="noopener">{esc(n)}{icon("arrow", 14)}</a>'
        for n, u in links.items()) + "</div>"


# ================= عدّاد الزوار (مرة واحدة لكل جلسة) =================
if "visit_counted" not in st.session_state:
    try:
        visitors.register_visit()
    except OSError:
        pass
    st.session_state.visit_counted = True
VISITS = visitors.get_counts()


def visitors_badge() -> str:
    """العداد يظهر بالإنجليزية دائماً، بغض النظر عن لغة الواجهة."""
    return (f'<div class="cv-visits" dir="ltr"><i></i><b>{VISITS["total"]:,}</b> Visitors'
            f'<span>·</span>Today <b>{VISITS["today"]:,}</b></div>')


# ================= شريط التنقل العلوي =================
with st.container(key="topnav"):
    c_brand, c_nav, c_lang = st.columns([1.6, 1.6, 1], vertical_alignment="center")
    with c_brand:
        html_block(f'<div class="cv-brand"><img class="cv-uni-logo" src="{LOGO_MARK}" alt="{esc(T("uni_name"))}">'
                   f'<span class="cv-sep"></span><div class="cv-logo">CV</div><div><b>CV Agent</b>'
                   f'<small>{esc(T("brand_sub"))}</small></div></div>')
    with c_nav:
        page = st.segmented_control("page", [0, 1], default=st.session_state.page, key="nav_page",
                                    format_func=lambda i: T("nav")[i], label_visibility="collapsed")
        if page is not None and page != st.session_state.page:
            st.session_state.page = page
            st.rerun()
    with c_lang:
        lang = st.segmented_control("lang", ["ar", "en"], default=LANG, key="nav_lang",
                                    format_func={"ar": "عربي", "en": "EN"}.get, label_visibility="collapsed")
        if lang and lang != LANG:
            st.session_state.lang = lang
            st.rerun()

def _secret(name: str) -> str:
    """المفتاح من متغيرات البيئة (Colab) أو من Secrets في Streamlit Community Cloud."""
    value = os.environ.get(name, "")
    if not value:
        try:
            value = st.secrets.get(name, "")
        except Exception:  # noqa: BLE001 — لا يوجد ملف secrets محلياً
            value = ""
    return value


env_key = _secret("OPENAI_API_KEY")






# ================= صفحة التحليل =================
def analysis_page():
    # ---- البانر ----
    with st.container(key="hero"):
        c_text, c_steps = st.columns([2.2, 1], gap="large", vertical_alignment="center")
        with c_text:
            html_block(f'<div class="cv-hero-tags"><span class="cv-tag-hero">{icon("spark", 14)}{esc(T("tag"))}</span>'
                       f'<span class="cv-tag-hero uni">{esc(T("college"))}</span></div>'
                       f'<div class="cv-hero-title">{esc(T("title"))}</div>'
                       f'<p class="cv-hero-sub">{esc(T("subtitle_hero"))}</p>'
                       f'<div class="cv-hero-cta"><a class="cv-btn-white" href="#start">{icon("upload", 16)}'
                       f'{esc(T("cta_upload"))}</a>' + "".join(f'<span class="cv-stat">{esc(s)}</span>'
                                                            for s in T("hero_stats")) + '</div>')
        with c_steps:
            html_block(f'<div class="cv-steps"><h4>{esc(T("how_title"))}</h4>' + "".join(
                f'<div class="cv-step"><i>{i + 1}</i><div><b>{esc(a)}</b><span>{esc(b)}</span></div></div>'
                for i, (a, b) in enumerate(T("steps"))) + "</div>")

    # ---- لوحة التحكم ----
    html_block('<div id="start"></div>')
    with st.container(key="panel"):
        html_block(title_row(T("panel_title"), "upload"))
        api_key = env_key  # المفتاح من Secrets فقط، ولا توجد خانة له في الواجهة
        if not api_key:
            html_block(f'<div class="cv-info">{icon("info")}<span>{esc(T("no_key"))}</span></div>')
        model = DEFAULT_MODEL  # النموذج ثابت (يتغير فقط من Colab Secrets عبر OPENAI_MODEL) ولا يظهر للمستخدم
        c_up, c_city, c_btn = st.columns([3, 1.4, 1.1], vertical_alignment="bottom")
        # تسميات ثابتة للعناصر (label_visibility) حتى لا تُمسح قيمها عند تبديل اللغة
        with c_up:
            field_label(T("upload"))
            uploaded = st.file_uploader("cv_file", type=["pdf", "docx"], key="cv_file", label_visibility="collapsed")
        with c_city:
            field_label(T("city"))
            city = st.selectbox("city", list(CITIES.keys()), key="city", label_visibility="collapsed",
                                format_func=lambda k: city_name(k, LANG))
        with c_btn:
            clicked = st.button(T("analyze"), type="primary", disabled=not (uploaded and api_key),
                                use_container_width=True)

    if uploaded:
        data = uploaded.getvalue()
        run_key = (hashlib.sha256(data).hexdigest(), city, model, LANG)
        if clicked:
            try:
                cv_text = extract_text(data, uploaded.name)
                with st.status(T("analyzing"), expanded=True) as status:
                    result = analyze_cv(cv_text, city=city, lang=LANG,
                                        llm=build_llm(api_key=api_key, model=model), on_step=status.write)
                    status.update(label=T("done"), state="complete", expanded=False)
                st.session_state.update(result=result, cv_text=cv_text, run_key=run_key)
            except CVProcessingError as e:
                st.error(f"❌ {e.message(LANG)}")
            except Exception as e:  # noqa: BLE001
                name = type(e).__name__
                if "Authentication" in name:
                    st.error(f"❌ {T('err_auth')}")
                elif "RateLimit" in name:
                    st.error(f"❌ {T('err_rate')}")
                elif "NotFound" in name:
                    st.error(f"❌ {T('err_model', model=model)}")
                else:
                    st.error(f"❌ {T('err_unexpected')} {e}")
        stored = st.session_state.get("run_key")
        if st.session_state.get("result") and stored != run_key:
            st.info(T("stale_file") if stored[0] != run_key[0] else T("stale_settings"))

    r = st.session_state.get("result")
    if r:
        show_results(r)


def show_results(r):
    prio, level = t(LANG, "priority"), t(LANG, "level")
    c, g = r.candidate, r.skill_gap
    report_bytes = build_markdown_report(r).encode("utf-8")
    if r.truncated:
        st.warning(T("truncated"))

    top = f"{r.jobs.jobs[0].match_percentage}%" if r.jobs.jobs else "—"
    metrics = [("user", T("m_name"), c.name, ""), ("brief", T("m_years"), f"{c.experience_years:g}", ""),
               ("list", T("m_skills"), len(c.extracted_skills), ""), ("target", T("m_top"), top, " accent")]
    html_block('<div class="cv-grid">' + "".join(
        f'<div class="cv-card cv-metric"><div class="cv-ic{cls}">{icon(ic)}</div><div>'
        f'<div class="cv-label">{esc(l)}</div><div class="cv-value{cls}">{esc(v)}</div></div></div>'
        for ic, l, v, cls in metrics) + "</div>")

    def gap_html() -> str:
        return "".join(
            f'<div class="cv-gap-item"><span class="cv-badge {s.priority}">{esc(prio[s.priority])}</span>'
            f'<div><b>{esc(s.skill)}</b><span class="cv-sub">{esc(s.reason)}</span></div></div>'
            for s in g.missing_skills)

    tab_profile, tab_gap, tab_jobs, tab_courses, tab_cv = st.tabs(T("tabs"))

    with tab_profile:
        html_block(
            '<div class="cv-card">'
            f'<div class="cv-label">{esc(T("current_title"))}</div><div class="cv-h3">{esc(c.current_title)}</div><br>'
            f'<div class="cv-label">{esc(T("field"))}</div><div class="cv-h3">{esc(c.field)}</div><br>'
            f'<div class="cv-label">{esc(T("education"))}</div><div class="cv-h3">{esc(c.education)}</div>'
            f'<p class="cv-text">{esc(c.summary)}</p></div>'
            f'<div class="cv-card">{title_row(T("skills"), "list")}'
            + "".join(f'<span class="cv-tag">{esc(s)}</span>' for s in c.extracted_skills) + "</div>")
        with st.expander(T("extracted_text")):
            st.text(st.session_state.get("cv_text", ""))

    with tab_gap:
        html_block(f'<div class="cv-card">{title_row(T("gap_heading"), "target")}{gap_html()}</div>'
                   f'<div class="cv-card">{title_row(T("tips_heading"), "spark")}<ul class="cv-list">'
                   + "".join(f"<li>{esc(x)}</li>" for x in g.recommendations_to_improve) + "</ul></div>")

    with tab_jobs:
        col_jobs, col_side = st.columns([2.5, 1], gap="large")
        with col_jobs:
            html_block(f'<div class="cv-h2">{esc(T("jobs_heading", city=city_name(r.city, LANG)))}</div>'
                       f'<p class="cv-note">{esc(T("jobs_note"))}</p>')
            for j in r.jobs.jobs:
                sub = j.job_title_local if r.lang == "ar" else f'{city_name(r.city, "en")}, Saudi Arabia'
                html_block(
                    '<div class="cv-card"><div class="cv-top">'
                    f'<div><div class="cv-h3">{esc(j.job_title)}</div><div class="cv-sub">{esc(sub)}</div></div>'
                    f'<div class="cv-match"><b>{j.match_percentage}%</b><span>{esc(T("match"))}</span></div></div>'
                    f'<div class="cv-progress"><div style="width:{j.match_percentage}%"></div></div>'
                    f'<p class="cv-text">{esc(j.why_fit)}</p>'
                    f'<div class="cv-sub">{esc(T("employers"))} {esc(", ".join(j.example_employers))}</div>'
                    f'{chips(job_links(j.job_title, r.city, LANG))}</div>')
        with col_side:
            html_block(f'<div class="cv-card">{title_row(T("gap_heading"), "target")}{gap_html()}</div>')
            st.download_button(f"⬇ {T('download')}", data=report_bytes, file_name="cv_report.md",
                               mime="text/markdown", key="dl_jobs")

    with tab_courses:
        html_block(f'<div style="margin-top:12px">{title_row(T("courses_heading"), "book")}</div>')
        for co in r.courses.courses:
            html_block(
                '<div class="cv-card"><div class="cv-top">'
                f'<div><div class="cv-h3">{esc(co.course_name)}</div>'
                f'<div class="cv-sub">{esc(T("covers"))} {esc(co.target_skill)}</div></div>'
                f'<span class="cv-badge level">{esc(level[co.level])}</span></div>'
                f'{chips(course_links(co.course_name))}</div>')

    with tab_cv:
        imp = r.improved_cv
        st.text_area(T("summary_label"), imp.improved_summary, height=150)
        html_block(
            f'<div class="cv-card">{title_row(T("bullets_heading"), "list")}'
            f'<p class="cv-note">{esc(T("bullets_note"))}</p><ul class="cv-list" dir="auto">'
            + "".join(f'<li dir="auto">{esc(p)}</li>' for p in imp.improved_bullet_points) + "</ul></div>"
            f'<div class="cv-card">{title_row(T("ats_heading"), "spark")}<ul class="cv-list">'
            + "".join(f"<li>{esc(x)}</li>" for x in imp.ats_tips) + "</ul></div>")
        st.download_button(f"⬇ {T('download')}", data=report_bytes, file_name="cv_report.md",
                           mime="text/markdown", key="dl_cv")


# ================= صفحة فحص النظام =================
def check_page():
    model = DEFAULT_MODEL
    api_key = env_key
    with st.container(key="chk_header"):
        c_title, c_btn = st.columns([4, 1], vertical_alignment="center")
        with c_title:
            html_block(f'<p class="cv-chk-title">{esc(T("chk_title"))}</p>'
                       f'<p class="cv-sub" style="margin:4px 0 0">{esc(T("chk_sub"))}</p>')
        with c_btn:
            run = st.button(T("chk_run"), type="primary", use_container_width=True)
        if run:
            with st.spinner(T("chk_run") + "…"):
                st.session_state.checks = checks.run_all(LANG, api_key=api_key, model=model)
        results = st.session_state.get("checks")
        if results:
            passed = sum(r.status == "pass" for r in results)
            failed = sum(r.status == "fail" for r in results)
            total_s = sum(r.seconds for r in results)
            stats = [(T("chk_total"), len(results), "var(--ink)"), (T("chk_passed"), passed, "#15803D"),
                     (T("chk_failed"), failed, "#DC2626"), (T("chk_time"), f'{total_s:.2f} {T("chk_sec")}', "var(--blue)")]
            bar_color = "linear-gradient(90deg,var(--blue),#22C55E)" if not failed else "linear-gradient(90deg,#F59E0B,#DC2626)"
            html_block('<div class="cv-grid" style="margin:18px 0 0">' + "".join(
                f'<div class="cv-mini"><div class="cv-label">{esc(l)}</div>'
                f'<div class="cv-value" style="color:{col}">{esc(v)}</div></div>' for l, v, col in stats)
                + f'</div><div class="cv-chk-bar" style="background:{bar_color}"></div>')

    results = st.session_state.get("checks")
    if not results:
        html_block(f'<div class="cv-info">{icon("info")}<span>{esc(T("chk_empty"))}</span></div>')
    else:
        names, labels = T("chk_names"), T("chk_status")
        icons = {"pass": "check", "fail": "x", "skip": "minus"}
        html_block('<div class="cv-card" style="padding:8px 24px;margin-top:20px">' + "".join(
            f'<div class="cv-chk-row"><div class="st {r.status}">{icon(icons[r.status], 18)}</div>'
            f'<div class="tx"><b>{i + 1}. {esc(names[r.key])}</b><span class="cv-sub">{esc(r.detail)}</span></div>'
            f'<span class="tm">{icon("clock", 14)} {r.seconds:.2f} {esc(T("chk_sec"))}</span>'
            f'<span class="cv-badge {r.status}">{esc(labels[r.status])}</span></div>'
            for i, r in enumerate(results)) + "</div>")
    html_block(f'<div class="cv-info">{icon("info")}<span>{esc(T("chk_note"))}</span></div>')


# ================= التذييل =================
def footer():
    html_block(f'<div class="cv-footer"><div class="cv-f-main"><img src="{LOGO_WHITE}" alt="{esc(T("uni_name"))}">'
               f'<div><b>{esc(T("uni_name"))}</b><span>{esc(T("college"))}</span></div></div>'
               f'<div class="cv-f-meta">{visitors_badge()}<span>{esc(T("footer_project"))}</span>'
               f'<span>{esc(T("footer_sup"))} · {esc(T("footer_year"))}</span></div></div>')


if st.session_state.page == 1:
    check_page()
else:
    analysis_page()
footer()


## 13) تشغيل الواجهة
تظهر الواجهة **داخل الدفتر مباشرة** تحت الخلية، مع زر «افتح الواجهة» لنافذة كاملة. وللنشر الدائم برابط عام، المشروع جاهز للرفع على Streamlit Community Cloud.

In [12]:
import os, socket, subprocess, sys, time
from IPython.display import HTML, display
from google.colab import output

PORT = 8501

def wait_for_port(port, timeout=60):
    """ينتظر حتى يصبح السيرفر جاهزاً فعلاً بدل الانتظار لمدة ثابتة."""
    start = time.time()
    while time.time() - start < timeout:
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", port)) == 0:
                return True
        time.sleep(1)
    return False

def open_button(url, label):
    display(HTML(f'<a href="{url}" target="_blank" style="display:inline-block;margin:6px 8px 6px 0;'
                 f'padding:12px 22px;border-radius:12px;background:linear-gradient(90deg,#1D4ED8,#3B82F6);'
                 f'color:#fff;font-weight:600;text-decoration:none;font-family:sans-serif">{label} ↗</a>'))

# 1) إيقاف أي تشغيل سابق
subprocess.run(["pkill", "-f", "streamlit run"], check=False)
time.sleep(2)

# 2) تشغيل Streamlit مع حفظ السجل في ملف
log = open("streamlit.log", "w")
proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", str(PORT),
     "--server.headless", "true", "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
    env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT,
)
if not wait_for_port(PORT):
    proc.kill()
    print("❌ لم يبدأ السيرفر. آخر سطور السجل:")
    print(open("streamlit.log").read()[-2000:])
    raise SystemExit

print("✅ الواجهة جاهزة")

# 3) زر يفتح الواجهة في نافذة جديدة عبر Colab مباشرة
open_button(output.eval_js(f"google.colab.kernel.proxyPort({PORT})"), "افتح الواجهة")

# 4) عرض الواجهة داخل الدفتر مباشرة
output.serve_kernel_port_as_iframe(PORT, height=900)

✅ الواجهة جاهزة


ℹ️ رابط ngrok المجاني يعرض صفحة تحذير أول مرة: اضغط Visit Site.


<IPython.core.display.Javascript object>